# **RF_Classification.ipynb**

# Retrieve ARL Folder

In [ ]:
!mkdir -p ARL
base="https://github.com/James-Tiny-Tjib/ARL/raw/main/ARL"
!curl -L -o "ARL/Airport_Noise_Dataset.zip" "$base/Airport%20Noise%20Dataset-20260603T141901Z-3-001.zip"
!curl -L -o "ARL/DroneAudioDataset-20260603T141904Z-3-001.zip" "$base/DroneAudioDataset-20260603T141904Z-3-001.zip"
!curl -L -o "ARL/best_cnn.pt" "$base/best_cnn.pt"
!curl -L -o "ARL/dca.png" "$base/dca.png"
!curl -L -o "ARL/dca_plain.png" "$base/dca_plain.png"
!curl -L -o "ARL/drone.png" "$base/drone.png"
!curl -L -o "ARL/drone_demo_dataset.zip" "$base/drone_demo_dataset.zip"
!curl -L -o "ARL/drone_multi_classifier.pt" "$base/drone_multi_classifier.pt"
!curl -L -o "ARL/resnet50_drone_weights.pth" "$base/resnet50_drone_weights.pth"
!curl -L -o "ARL/sensor_client.pkl" "$base/sensor_client.pkl"

## Install Dependencies

In [ ]:
import torch as t
from torch.utils.data import TensorDataset
from torch.utils.data import Dataset, DataLoader
import torchvision.datasets as datasets
from sklearn.model_selection import train_test_split
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import pickle
import numpy as np
import os
import base64

from itertools import cycle
import copy
from collections import defaultdict
from torch.utils.data import ConcatDataset

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# IQCNN

In [ ]:


def save_data(data_var,filename):
  with open(filename, 'wb') as file:
    # Use pickle.dump() to write the variable to the file
    pickle.dump(data_var, file)

def read_data(filename):
  with open(filename, 'rb') as f:
        # Load the object from the pickle file
        loaded_data = pickle.load(f)
  return loaded_data

class IQCNN(nn.Module):
    def __init__(self, num_classes=11):
        super(IQCNN, self).__init__()

        self.layer_dims=[]
        self.layers=nn.ModuleList()

        # Define convolutional layers here .......................................
        self.layers.append(nn.Conv1d(in_channels=2, out_channels=8, kernel_size=7, padding=3,bias=False))
        self.layers.append(nn.Conv1d(in_channels=8, out_channels=16, kernel_size=7, padding=3,bias=False))
        self.layers.append(nn.Conv1d(in_channels=16, out_channels=32, kernel_size=7, padding=3,bias=False))
        self.layers.append(nn.Conv1d(in_channels=32, out_channels=64, kernel_size=7, padding=3,bias=False))

        self.conv_num=len(self.layers)
        for i in range(self.conv_num):
          in_ch=self.layers[i].in_channels
          ker_sz=self.layers[i].kernel_size[0]
          self.layer_dims.append(in_ch*ker_sz)

        #Define linear layers here .....................................................
        self.layers.append(nn.Linear(64, 256,bias=False))
        self.layers.append(nn.Linear(256, num_classes,bias=False))

        for i in range(self.conv_num,len(self.layers)):
          self.layer_dims.append(self.layers[i].in_features)
        self.layer_dims.append(num_classes)
        # print(self.layer_dims)

        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)

        for i in range(self.conv_num):
          nn.init.xavier_uniform_(self.layers[i].weight)
        for i in range(self.conv_num,len(self.layers)-1):
            nn.init.kaiming_normal_(self.layers[i].weight, nonlinearity='relu')

        nn.init.kaiming_normal_(self.layers[i].weight, nonlinearity='linear')

    def forward(self, x):

        for i in range(self.conv_num):
         x=F.relu(self.layers[i](x))

        x = self.global_avg_pool(x)  # (batch_size, 64, 1)
        x = x.squeeze(-1)  # (batch_size, 64)

        for i in range(self.conv_num,len(self.layers)-1):
          x=F.relu(self.layers[i](x))
        x = self.layers[-1](x)  # (batch_size, output_shape)

        return x#F.log_softmax(x, dim=1)  # Use log_softmax for classification

def build_model(num_classes=3,model_name='IQCNN'):
  return IQCNN(num_classes=num_classes)#IQCNN(num_classes=11)

#.............................Client Class......................................

class Client:
  def __init__(self,dataset,num_classes=3,device='cpu'):
      self.device=device
      self.model = build_model(num_classes=num_classes).to(self.device)
      self.dataset =dataset
      self.dataloader = cycle(t.utils.data.DataLoader(self.dataset, batch_size=512, shuffle=True))
      self.cross_loss = nn.CrossEntropyLoss()
      self.optimizer = t.optim.Adam(self.model.parameters(), lr=0.001)

  def load_model(self,state_dict):
    self.model.load_state_dict(state_dict)

  def load_model_from_file(self,filename):
    state_dict=read_data(filename)
    self.load_model(state_dict)

  def save_model(self,filename):
    save_data(self.model.state_dict,filename)

  def client_loss(self,pred,y):
    return self.cross_loss(pred,y)

  def evaluate(self,test_loader):
    correct,total=0,0
    with t.no_grad():
        for data in test_loader:
            x, y = data
            x=x.to(self.device)
            y=y.to(self.device)
            output = self.model(x)
            for idx, i in enumerate(output):
                if t.argmax(i) == y[idx]:
                    correct +=1
                total +=1
    print(f'accuracy: {round(correct/total, 3)}')
    return round(correct/total, 3)


  def train_batch(self):
    x,y = next(self.dataloader)
    x=x.to(self.device)
    y=y.to(self.device)
    self.optimizer.zero_grad()
    pred = self.model(x)
    loss=self.client_loss(pred,y)
    loss.backward()
    self.optimizer.step()
    return loss.item()

  def train(self,epochs=int(1e3),echo=True,eval_acc=False):
    running_loss=0
    for epoch in range(epochs):
      epoch_loss = self.train_batch()
      running_loss+=epoch_loss

      if echo:
        if epoch%int(epochs/10) ==0 and epoch!=0:
          print("epoch:"+str(epoch)+" loss:"+ str(running_loss/int(epochs/10)))
          running_loss=0
          if eval_acc:
            self.evaluate(test_loader)
        # if epoch%(echo*10)==0 and epoch!=0:
        #   self.evaluate()

    return running_loss/int(epochs/10)

def create_synthetic_data(noise_level,num_datapoints=int(1e4)):
  fc = 40
  samples = 32
  duration = 0.1
  t_sym = np.linspace(0, duration, samples, endpoint=False)
  num_symbols=128

  # Scales for Unit Power
  scale_qpsk = 1.0 / np.sqrt(2)
  scale_16qam = 1.0 / np.sqrt(10)

  mod_options = ['BPSK', 'QPSK', '16-QAM']
  label_dict={'BPSK':0, 'QPSK':1, '16-QAM':2}
  data=[]
  labels=[]
  for _ in range(num_datapoints):
    m=np.random.choice(mod_options, 1)
    m=m[0]
    ground_truth = [m]*4 #128/4

    rx_i, rx_q = np.array([]), np.array([])
    for idx, mod in enumerate(ground_truth):
        # -- Normalized Transmitter --
        if mod == 'BPSK':
            s = (2*np.random.randint(0,2)-1) + 0j
        elif mod == 'QPSK':
            raw = (2*np.random.randint(0,2)-1) + 1j*(2*np.random.randint(0,2)-1)
            s = raw * scale_qpsk
        else:
            mp = np.array([-3, -1, 1, 3])
            raw = mp[np.random.randint(0,4)] + 1j*mp[np.random.randint(0,4)]
            s = raw * scale_16qam

        # Channel
        raw_wave = (s.real * np.cos(2*np.pi*fc*t_sym) - s.imag * np.sin(2*np.pi*fc*t_sym))

        # Add noise (Standard deviation)
        noise = noise_level * np.random.randn(len(raw_wave))
        noisy_chunk = raw_wave + noise

        # Rx - Match filter output
        rec_i = noisy_chunk * 2 * np.cos(2*np.pi*fc*t_sym)
        rec_q = noisy_chunk * -2 * np.sin(2*np.pi*fc*t_sym)
        rx_i=np.concatenate((rx_i,rec_i)); rx_q=np.concatenate((rx_q,rec_q))
    data.append(np.array([rx_i,rx_q]))
    labels.append(label_dict[m])

  X=np.array(data)
  Y=np.array(labels)
  print(X.shape,Y.shape)

  X_train_ar, X_test_ar, y_train_ar, y_test_ar = train_test_split(X, Y, test_size=0.3, random_state=42, stratify=Y) # split the data (70% train, 30% test)
  print(X_train_ar.shape, X_test_ar.shape, y_train_ar.shape, y_test_ar.shape)

  X_train = t.from_numpy(X_train_ar).float()
  y_train = t.from_numpy(y_train_ar).long() # Use long for integer labels
  X_test = t.from_numpy(X_test_ar).float()
  y_test = t.from_numpy(y_test_ar).long()
  trainset = TensorDataset(X_train, y_train)
  testset = TensorDataset(X_test, y_test)

  batch_size = 512

  train_loader = t.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)
  test_loader = t.utils.data.DataLoader(testset, batch_size=10, shuffle=True)

  return train_loader,test_loader,trainset,testset



In [ ]:
# @title create synthetic RF data
train_loader,test_loader,trainset,testset=create_synthetic_data(noise_level=0.1,num_datapoints=int(1e4))
device= 'cuda' if t.cuda.is_available() else 'cpu'
print(device)

(10000, 2, 128) (10000,)
(7000, 2, 128) (3000, 2, 128) (7000,) (3000,)
cuda


In [ ]:
# @title train RF client
rf_client= Client(trainset,device=device)
rf_client.train(epochs=int(5e3),echo=True,eval_acc=True)
rf_client.evaluate(test_loader)

epoch:500 loss:0.13799876813590525
accuracy: 0.986
epoch:1000 loss:0.041664256968535485
accuracy: 0.993
epoch:1500 loss:0.018177514124894514
accuracy: 0.996
epoch:2000 loss:0.014300265724305063
accuracy: 0.997
epoch:2500 loss:0.012398271567653864
accuracy: 0.997
epoch:3000 loss:0.01196321437950246
accuracy: 0.998
epoch:3500 loss:0.011487901421263814
accuracy: 0.997
epoch:4000 loss:0.010818639640696346
accuracy: 0.997
epoch:4500 loss:0.15026226408570073
accuracy: 0.98
accuracy: 0.987


0.987

In [ ]:
rf_client.save_model('saved_data.pkl')

PicklingError: Can't pickle <class '__main__.Client'>: it's not the same object as __main__.Client

In [ ]:
rf_client = read_data("saved_data.pkl")
rf_client.evaluate(test_loader)

# Composite Signal Classification

In [ ]:
import itertools

def create_composite_data(noise_level,user_mods,num_datapoints=int(1e4)):
  fc = 40
  samples = 32
  duration = 0.1
  t_sym = np.linspace(0, duration, samples, endpoint=False)
  num_symbols=4


  # Scales for Unit Power
  scale_qpsk = 1.0 / np.sqrt(2)
  scale_16qam = 1.0 / np.sqrt(10)

  mod_options = ['BPSK', 'QPSK', '16-QAM']

  product_iterator=itertools.product(*user_mods)
  label_list=list(product_iterator)
  label_dict={'BPSK':0, 'QPSK':1, '16-QAM':2}
  label_dict={}
  for i,l in enumerate(label_list):
    label_dict[str(list(l))]=i
  num_classes=len(label_dict)
  data=[]
  labels=[]
  for dpt in range(num_datapoints):
    # print(dpt)
    composite_label=[]
    for u in range(len(user_mods)):
      m=np.random.choice(user_mods[u], 1)
      m=str(m[0])
      composite_label.append(m)

    ground_truth = [composite_label]*num_symbols #128/4
    rx_i, rx_q = np.array([]), np.array([])

    for idx, mods in enumerate(ground_truth):
        # -- Normalized Transmitter --
        s=0+0j
        for mod in mods:
          if mod == 'BPSK':
              s += (2*np.random.randint(0,2)-1) + 0j
          elif mod == 'QPSK':
              raw = (2*np.random.randint(0,2)-1) + 1j*(2*np.random.randint(0,2)-1)
              s += raw * scale_qpsk
          else:
              mp = np.array([-3, -1, 1, 3])
              raw = mp[np.random.randint(0,4)] + 1j*mp[np.random.randint(0,4)]
              s += raw * scale_16qam


        # Channel
        raw_wave = (s.real * np.cos(2*np.pi*fc*t_sym) - s.imag * np.sin(2*np.pi*fc*t_sym))

        # Add noise (Standard deviation)
        noise = noise_level * np.random.randn(len(raw_wave))
        noisy_chunk = raw_wave + noise

        # Rx - Match filter output
        rec_i = noisy_chunk * 2 * np.cos(2*np.pi*fc*t_sym)
        rec_q = noisy_chunk * -2 * np.sin(2*np.pi*fc*t_sym)

        rx_i=np.concatenate((rx_i,rec_i)); rx_q=np.concatenate((rx_q,rec_q))
        # rx_i=np.concatenate((rx_i,np.array([rec_i]))); rx_q=np.concatenate((rx_q,np.array([rec_q])))
    data.append(np.array([rx_i,rx_q]))
    labels.append(label_dict[str(composite_label)])

  X=np.array(data)
  Y=np.array(labels)
  print(X.shape,Y.shape)

  X_train_ar, X_test_ar, y_train_ar, y_test_ar = train_test_split(X, Y, test_size=0.3, random_state=42, stratify=Y) # split the data (70% train, 30% test)
  print(X_train_ar.shape, X_test_ar.shape, y_train_ar.shape, y_test_ar.shape)

  X_train = t.from_numpy(X_train_ar).float()
  y_train = t.from_numpy(y_train_ar).long() # Use long for integer labels
  X_test = t.from_numpy(X_test_ar).float()
  y_test = t.from_numpy(y_test_ar).long()
  trainset = TensorDataset(X_train, y_train)
  testset = TensorDataset(X_test, y_test)

  batch_size = 512

  train_loader = t.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)
  test_loader = t.utils.data.DataLoader(testset, batch_size=10, shuffle=True)

  return train_loader,test_loader,trainset,testset,num_classes,label_list

In [ ]:
user_mods=[['BPSK','QPSK'],
           ['BPSK','QPSK','16-QAM']]
print(user_mods)
train_loader,test_loader,trainset,testset,num_composite_classes,label_list=create_composite_data(noise_level=0.1,user_mods=user_mods,num_datapoints=int(1e4))
device= 'cuda' if t.cuda.is_available() else 'cpu'
print(device)

In [ ]:
rf_client= Client(trainset,num_classes=num_composite_classes,device=device)
rf_client.train(epochs=int(1e4),echo=True,eval_acc=True)
rf_client.evaluate(test_loader)

In [ ]:
x=trainset[100]
print(x[0].shape)
y=x[1]
xx=t.unsqueeze(x[0],dim=0).to(device)
pred=rf_client.model(xx)
print(pred,t.argmax(pred))
print(y)

# Modulation Classification (OLD)

The window size is fixed right now to 4 symbols\
samples per symbol fixed at 32

In [ ]:
# 1. Install Gradio
!pip install gradio -q

import gradio as gr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import time

# --- Global State ---
class SignalState:
    def __init__(self):
        self.running = False
        self.samples_per_symbol = 32
        self.num_symbols = 20
        self.symbol_duration = 0.1
        self.noise_level = 0.05 # Lower default noise for QAM visibility
        self.waveform_segments = []
        self.ground_truth_mods = []
        self.predicted_mods = []
        self.rx_i = []
        self.rx_q = []

        self.filtered_chunks=[]

sim_state = SignalState()

# --- 1. CLASSIFIER (Normalized) ---
def classify_normalized(i_val, q_val):
    point = i_val + 1j*q_val

    # --- Define NORMALIZED Constellations (Avg Power = 1.0) ---
    # BPSK: +/- 1
    # Power = 1^2 = 1.
    bpsk_pts = [-1, 1]

    # QPSK: (+/- 1 +/- 1j) / sqrt(2)
    # Mag = sqrt(1^2 + 1^2) = sqrt(2). Divided by sqrt(2) = 1.
    scale_qpsk = 1.0 / np.sqrt(2)
    qpsk_pts = [scale_qpsk*(-1-1j), scale_qpsk*(-1+1j), scale_qpsk*(1-1j), scale_qpsk*(1+1j)]

    # 16-QAM: (+/-1, +/-3) / sqrt(10)
    # Avg Power of unscaled 16-QAM is 10. So we divide by sqrt(10).
    scale_16qam = 1.0 / np.sqrt(10)
    vals = [-3, -1, 1, 3]
    qam_pts = [scale_16qam*(x + 1j*y) for x in vals for y in vals]

    # --- Euclidean Distances ---
    err_bpsk = min([abs(point - p) for p in bpsk_pts])
    err_qpsk = min([abs(point - p) for p in qpsk_pts])
    err_16qam = min([abs(point - p) for p in qam_pts])

    # --- Penalties ---
    # Since 16-QAM points are now VERY close together (dense),
    # the penalty is crucial to avoid false positives from noise.
    scores = {
        'BPSK': err_bpsk + 0.0,
        'QPSK': err_qpsk + 0.1,
        '16-QAM': err_16qam + 0.25
    }

    return min(scores, key=scores.get)

# --- 2. Signal Generation ---
def generate_frame(noise_level):
    fc = 40
    samples = sim_state.samples_per_symbol
    duration = sim_state.symbol_duration
    t_sym = np.linspace(0, duration, samples, endpoint=False)

    # Scales for Unit Power
    scale_qpsk = 1.0 / np.sqrt(2)
    scale_16qam = 1.0 / np.sqrt(10)

    mod_options = ['BPSK', 'QPSK', '16-QAM']
    split1, split2 = np.random.randint(4, 8), np.random.randint(12, 16)
    m1, m2, m3 = np.random.choice(mod_options, 3)
    ground_truth = [m1]*split1 + [m2]*(split2-split1) + [m3]*(20-split2)

    waveform_segments = []
    filtered_segments=[]
    predictions = []
    rx_i, rx_q = [], []

    for idx, mod in enumerate(ground_truth):
        # -- Normalized Transmitter --
        if mod == 'BPSK':
            s = (2*np.random.randint(0,2)-1) + 0j
        elif mod == 'QPSK':
            raw = (2*np.random.randint(0,2)-1) + 1j*(2*np.random.randint(0,2)-1)
            s = raw * scale_qpsk
        else:
            mp = np.array([-3, -1, 1, 3])
            raw = mp[np.random.randint(0,4)] + 1j*mp[np.random.randint(0,4)]
            s = raw * scale_16qam

        # Channel
        raw_wave = (s.real * np.cos(2*np.pi*fc*t_sym) - s.imag * np.sin(2*np.pi*fc*t_sym))

        # Add noise (Standard deviation)
        noise = noise_level * np.random.randn(len(raw_wave))
        noisy_chunk = raw_wave + noise

        # Rx
        rec_i = np.mean(noisy_chunk * 2 * np.cos(2*np.pi*fc*t_sym))
        rec_q = np.mean(noisy_chunk * -2 * np.sin(2*np.pi*fc*t_sym))

        # Classify
        pred_mod = classify_normalized(rec_i, rec_q)
        predictions.append(pred_mod)
        rx_i.append(rec_i); rx_q.append(rec_q)

        # Color
        color = '#FF3333' if pred_mod == '16-QAM' else '#00FF00'

        start_t = idx * duration
        t_chunk = np.linspace(start_t, (idx+1)*duration, samples)
        waveform_segments.append( (t_chunk, noisy_chunk, color) )
        filtered_segments.append(np.array([noisy_chunk * 2 * np.cos(2*np.pi*fc*t_sym),noisy_chunk * -2 * np.sin(2*np.pi*fc*t_sym)]))

    sim_state.waveform_segments = waveform_segments
    sim_state.filtered_chunks=filtered_segments
    sim_state.predicted_mods = predictions
    sim_state.rx_i = rx_i
    sim_state.rx_q = rx_q

# --- 3. Visualization ---
def plot_signal(scan_start=None, scan_width=None):
    if not sim_state.waveform_segments: generate_frame(0.1)

    plt.style.use('dark_background')
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    fig.patch.set_facecolor('#121212')

    # Waveform
    for t_chunk, wave_chunk, color in sim_state.waveform_segments:
        ax1.plot(t_chunk, wave_chunk, color=color, lw=1.2)

    # Highlight
    result_text = "System Ready"
    if scan_start is not None:

        rect = patches.Rectangle((scan_start, -6), scan_width, 12,
                                 linewidth=2, edgecolor='yellow', facecolor='yellow', alpha=0.3)
        ax1.add_patch(rect)
        start_idx = int(scan_start / 0.1)
        end_idx = int((scan_start + scan_width) / 0.1)
        preds = sim_state.predicted_mods[start_idx : end_idx+1]
        unique_preds = sorted(list(set(preds)))
        # mod_txt = ", ".join([f"<b style='color:{'red' if '16-QAM' in m else '#00FF00'}'>{m}</b>" for m in unique_preds])
        # result_text = f"AI Output: {mod_txt}"

        # rf client ..............................................................................................
        # rect2 = patches.Rectangle((scan_start, -6), 4*0.1, 12,
        #                          linewidth=2, edgecolor='blue', facecolor='blue', alpha=0.3)
        # ax1.add_patch(rect2)
        start_idx_2=int(scan_start / 0.1)
        end_idx_2=start_idx_2+4 #128/32
        # noisy_signal=np.hstack(sim_state.filtered_chunks[start_idx_2:end_idx_2])
        # # print(noisy_signal.shape)
        # inp=t.from_numpy(noisy_signal)
        # inp=t.unsqueeze(inp,0).to(device)
        # inp=inp.to(t.float32)
        # # result_text = f"AI Output: {str(inp.dtype)}"
        # iqcnn_preds= t.argmax(rf_client.model(inp))
        # mod_options = ['BPSK', 'QPSK', '16-QAM']
        # iqcnn_preds = mod_options[int(iqcnn_preds)]
        noisy_signal=np.hstack(sim_state.filtered_chunks[start_idx:end_idx+1])
        num_frames = (noisy_signal.shape[1]) // 4
        #print(num_frames)
        iqcnn_preds=[]
        for i in range(num_frames):
          start =  i
          end = start + 4
          chunk = noisy_signal[:, start:end]
          inp = t.from_numpy(chunk)
          inp=t.unsqueeze(inp,0).to(device)
          inp=inp.to(t.float32)

          iqcnn_pred = t.argmax(rf_client.model(inp), dim = -1)
          mod_options = ['BPSK', 'QPSK', '16-QAM']
          iqcnn_preds.append(mod_options[int(iqcnn_pred)])
        unique_iqcnn_preds = sorted(list(set(iqcnn_preds)))
        mod_txt1 = ", ".join([f"<b style='color:{'red' if '16-QAM' in m else '#00FF00'}'>{m}</b>" for m in unique_iqcnn_preds])
        mod_txt2 = ", ".join([f"<b style='color:{'red' if '16-QAM' in m else '#00FF00'}'>{m}</b>" for m in unique_preds])
        result_text = f"Detected Mod via IQCNN: {mod_txt1}, Detected Mod via MaxL: {mod_txt2}"
        #..........................................................................................................

    ax1.set_title("Time Domain (Normalized Power)", fontsize=14, color='white')
    ax1.set_xlim(0, 2.0); ax1.set_ylim(-3, 3)
    ax1.grid(True, alpha=0.2)

    # Constellation
    # Draw Reference Boxes to see the "Density" difference
    # QPSK Box
    ax2.add_patch(patches.Rectangle((-0.8, -0.8), 1.6, 1.6, fill=False, edgecolor='green', linestyle='--', alpha=0.5, label='QPSK Region'))
    # 16-QAM Box (It's bigger overall, but points are denser)
    ax2.add_patch(patches.Rectangle((-1.1, -1.1), 2.2, 2.2, fill=False, edgecolor='red', linestyle=':', alpha=0.5, label='16-QAM Bounds'))

    ax2.scatter(sim_state.rx_i, sim_state.rx_q, color='cyan', alpha=0.3, s=40)
    if scan_start is not None:
        sel_i = sim_state.rx_i[start_idx : end_idx+1]
        sel_q = sim_state.rx_q[start_idx : end_idx+1]
        sel_c = ['red' if m == '16-QAM' else '#00FF00' for m in preds]
        ax2.scatter(sel_i, sel_q, c=sel_c, edgecolors='white', s=200, marker='X', zorder=10)

    ax2.set_title("Constellation (Normalized)", fontsize=14, color='white')
    ax2.set_xlim(-2, 2); ax2.set_ylim(-2, 2)
    ax2.grid(True, alpha=0.2)
    ax2.legend(loc='lower right', fontsize=8)

    plt.tight_layout()
    return fig, result_text

# --- Interface ---
def stream(n, s):
    sim_state.running = True
    while sim_state.running:
        generate_frame(n)
        fig, _ = plot_signal(None, None)
        yield fig, "<h3>Scanning...</h3>", gr.update(visible=False), gr.update(visible=False)
        plt.close(fig); time.sleep(s)

def pause():
    sim_state.running = False
    return "<h3>Paused</h3>", gr.update(visible=True), gr.update(visible=True)

def analyze(s, w): return plot_signal(s, w)

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# ⚖️ Normalized Power Classifier")
    gr.Markdown("Here, QPSK and 16-QAM have the **same average power**. Notice how 16-QAM points are crowded together!")

    with gr.Row():
        with gr.Column(scale=1):
            noise = gr.Slider(0, 0.5, value=0.05, label="Noise (Low for QAM)")
            speed = gr.Slider(0.01, 1.0, value=0.1, label="Speed")
            btn_start = gr.Button("▶ Start", variant="primary")
            btn_pause = gr.Button("⏸ Pause", variant="secondary")
            out_txt = gr.HTML("<h3>Ready</h3>")
            s_start = gr.Slider(0, 1.9, 0.1, label="Time Start", visible=False)
            s_width = gr.Slider(0.1, 1.0, 0.2, label="Width", visible=False)

        with gr.Column(scale=3): plot = gr.Plot()

    btn_start.click(stream, [noise, speed], [plot, out_txt, s_start, s_width])
    btn_pause.click(pause, None, [out_txt, s_start, s_width])
    s_start.change(analyze, [s_start, s_width], [plot, out_txt])
    s_width.change(analyze, [s_start, s_width], [plot, out_txt])

generate_frame(0.05)
if __name__ == "__main__": demo.queue().launch(share=True)

# Frequency Hopping (OLD)


In [ ]:
# 1. Install Gradio
!pip install gradio -q

import gradio as gr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import time

# --- 1. CONFIGURATION ---
class SystemState:
    def __init__(self):
        self.fs = 1000
        self.chunk_duration = 4.0
        self.running = False
        self.current_waveform = np.array([])

        # PROFILES (Now with BPSK!)
        self.profiles = {
            "S1 (Long Range)": {
                "freqs": [30, 40],
                "mod": "BPSK",
                "color": "#00CCFF", # Cyan/Blue
                "desc": "Robust / Low Rate"
            },
            "S2 (Standard)": {
                "freqs": [80, 100],
                "mod": "QPSK",
                "color": "#00FF00", # Green
                "desc": "Standard Link"
            },
            "S3 (THREAT)": {
                "freqs": [150, 200, 250],
                "mod": "16-QAM",
                "color": "#FF0000", # Red
                "desc": "High Speed Burst"
            }
        }

sys_state = SystemState()

# --- 2. MIXED SIGNAL GENERATOR ---
def generate_mixed_chunk(noise_level):
    fs = sys_state.fs
    total_dur = sys_state.chunk_duration

    full_wave = []
    current_t = 0

    # Generate segments until buffer is full
    while current_t < total_dur:
        seg_dur = np.random.uniform(1.0, 2.0)
        if current_t + seg_dur > total_dur: seg_dur = total_dur - current_t

        # Pick Random Profile
        p_name = np.random.choice(list(sys_state.profiles.keys()))
        profile = sys_state.profiles[p_name]

        freq_list = profile["freqs"]
        mod_type = profile["mod"]

        # Generate Hopping
        hop_dur = 0.2
        num_hops = int(np.ceil(seg_dur / hop_dur))
        segment_wave = []

        for i in range(num_hops):
            this_hop_dur = min(hop_dur, seg_dur - (i*hop_dur))
            if this_hop_dur <= 0: break

            f_c = freq_list[i % len(freq_list)]
            t = np.linspace(0, this_hop_dur, int(fs*this_hop_dur), endpoint=False)

            # --- MODULATION LOGIC ---
            num_syms = max(5, int(20 * this_hop_dur / 0.2))

            if mod_type == "16-QAM":
                scale = 1.0/np.sqrt(10)
                levs = [-3, -1, 1, 3]
                syms = [complex(x,y)*scale for x,y in zip(
                    np.random.choice(levs, num_syms), np.random.choice(levs, num_syms))]

            elif mod_type == "QPSK":
                scale = 1.0/np.sqrt(2)
                syms = [(np.random.choice([-1,1]) + 1j*np.random.choice([-1,1]))*scale
                        for _ in range(num_syms)]

            else: # BPSK (New!)
                # BPSK is just +1 or -1 on the Real axis.
                # We often leave Imaginary as 0.
                syms = [(2*np.random.randint(0,2)-1) + 0j for _ in range(num_syms)]

            # Upsample & Upconvert
            s_up = np.repeat(syms, int(len(t)/num_syms) + 1)[:len(t)]
            wave = s_up.real * np.cos(2*np.pi*f_c*t) - s_up.imag * np.sin(2*np.pi*f_c*t)
            segment_wave.append(wave)

        if segment_wave: full_wave.append(np.concatenate(segment_wave))
        current_t += seg_dur

    raw = np.concatenate(full_wave)
    raw = raw[:int(fs*total_dur)]

    # Add Noise
    sys_state.current_waveform = raw + (np.random.randn(len(raw)) * noise_level)

# --- 3. DECODER ---
def analyze_selection(start_t, width_t):
    fs = sys_state.fs
    wave = sys_state.current_waveform

    idx_start = int(start_t * fs)
    idx_end = int((start_t + width_t) * fs)
    idx_start = max(0, min(idx_start, len(wave)-1))
    idx_end = max(0, min(idx_end, len(wave)-1))

    if idx_end - idx_start < 50: return None

    chunk = wave[idx_start:idx_end]

    # 1. Frequency (FFT)
    fft_vals = np.abs(np.fft.fft(chunk))
    fft_freqs = np.fft.fftfreq(len(chunk), 1/fs)
    pos = fft_freqs > 0

    peak_idx = np.argmax(fft_vals[pos])
    dom_freq = fft_freqs[pos][peak_idx]

    threshold = np.max(fft_vals[pos]) * 0.4
    peaks = fft_freqs[pos][fft_vals[pos] > threshold]
    detected_freqs = sorted(list(set([int(round(f/10)*10) for f in peaks if f > 10])))

    # 2. Match Profile
    best_match = "Unknown"
    best_score = 0
    match_color = "gray"

    for name, profile in sys_state.profiles.items():
        common = len(set(profile["freqs"]).intersection(set(detected_freqs)))
        if common > best_score:
            best_score = common
            best_match = name
            match_color = profile["color"]

    # 3. Demodulation
    t_chunk = np.linspace(0, width_t, len(chunk), endpoint=False)
    baseband = chunk * np.exp(-1j * 2 * np.pi * dom_freq * t_chunk)
    constellation = baseband[::45] * 2.0 # Scale for visibility

    # Auto-Rotate Phase Correction (Simple)
    # BPSK/QPSK often come out rotated due to phase offset.
    # We rotate the whole cloud so the mean angle aligns with 45 deg or 0 deg for cleaner viewing
    avg_angle = np.angle(np.mean(constellation**4))/4 # 4th power carrier recovery trick
    constellation = constellation * np.exp(-1j * avg_angle)

    return {
        "seq": best_match,
        "color": match_color,
        "iq": constellation,
        "freqs": detected_freqs,
        "dom_freq": dom_freq
    }

# --- 4. VISUALIZATION ---
def update_plot(scan_start=None, scan_width=None):
    if len(sys_state.current_waveform) == 0: generate_mixed_chunk(0.1)

    wave = sys_state.current_waveform
    fs = sys_state.fs

    fig = plt.figure(figsize=(14, 8))
    fig.patch.set_facecolor('#121212')
    plt.style.use('dark_background')

    gs = fig.add_gridspec(2, 2, width_ratios=[2, 1])
    ax_time = fig.add_subplot(gs[0, 0])
    ax_spec = fig.add_subplot(gs[1, 0])
    ax_iq = fig.add_subplot(gs[:, 1])

    # Standard Plots
    ax_time.plot(np.linspace(0, 4, len(wave)), wave, color='#00FF00', lw=0.5)
    ax_time.set_title("Time Domain", color='white')
    ax_time.set_ylim(-4, 4); ax_time.set_xlim(0, 4)

    ax_spec.specgram(wave, NFFT=256, Fs=fs, noverlap=128, cmap='inferno')
    ax_spec.set_title("Spectrogram", color='white')
    ax_spec.set_ylim(0, 300)

    # IQ Default
    ax_iq.set_title("Constellation Scope", color='white')
    ax_iq.set_xlim(-2.5, 2.5); ax_iq.set_ylim(-2.5, 2.5)
    ax_iq.grid(True, alpha=0.2)
    ax_iq.axhline(0, color='gray'); ax_iq.axvline(0, color='gray')
    ax_iq.text(0,0, "PAUSE & SELECT", ha='center', color='gray')

    status_html = "<h3>Scanning...</h3>"

    if scan_start is not None:
        # Highlight Box
        rect_s = patches.Rectangle((scan_start, 0), scan_width, 300, edgecolor='white', facecolor='white', alpha=0.1)
        ax_spec.add_patch(rect_s)

        res = analyze_selection(scan_start, scan_width)
        if res:
            ax_iq.clear()
            iq = res["iq"]
            col = res["color"]

            # Scatter
            ax_iq.scatter(iq.real, iq.imag, color=col, s=60, edgecolors='white', alpha=0.9)

            # Guides
            seq = res["seq"]
            if "BPSK" in seq or "Long Range" in seq:
                # BPSK: 2 dots on X axis
                ax_iq.add_patch(patches.Circle((-1,0), 0.3, fill=False, edgecolor='cyan', linestyle='--'))
                ax_iq.add_patch(patches.Circle((1,0), 0.3, fill=False, edgecolor='cyan', linestyle='--'))
                mod_name = "BPSK (2-Point)"
            elif "QAM" in seq:
                mod_name = "16-QAM (16-Point)"
            else:
                mod_name = "QPSK (4-Point)"

            ax_iq.set_title(f"Decoded: {mod_name}", color=col, fontsize=14, weight='bold')
            ax_iq.set_xlim(-2.5, 2.5); ax_iq.set_ylim(-2.5, 2.5)
            ax_iq.grid(True, alpha=0.2)

            status_html = f"<h3 style='color:{col}'>{seq}</h3>Detected Mod: {mod_name}"

    plt.tight_layout()
    return fig, status_html

# --- INTERFACE ---
def stream(noise):
    sys_state.running = True
    while sys_state.running:
        generate_mixed_chunk(noise)
        fig, _ = update_plot(None, None)
        yield fig, "<h3>Monitoring...</h3>", gr.update(visible=False), gr.update(visible=False)
        plt.close(fig); time.sleep(2.0)

def pause():
    sys_state.running = False
    return "<h3>Paused</h3>", gr.update(visible=True), gr.update(visible=True)

def analyze(s, w): return update_plot(s, w)

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📡 3-Modulation Interceptor")
    gr.Markdown("Detect **BPSK (Blue)**, **QPSK (Green)**, and **16-QAM (Red/Threat)**.")
    with gr.Row():
        with gr.Column(scale=1):
            noise = gr.Slider(0, 0.5, 0.1, label="Noise")
            btn_play = gr.Button("▶ Start", variant="primary")
            btn_pause = gr.Button("⏸ Pause", variant="secondary")
            out = gr.HTML("Ready")
            s_start = gr.Slider(0, 3.6, 0.0, step=0.2,label="Time", visible=False)
            s_width = gr.Slider(0.1, 1.5, 0.4,step=0.2, label="Width", visible=False)
        with gr.Column(scale=3): plot = gr.Plot()

    btn_play.click(stream, [noise], [plot, out, s_start, s_width])
    btn_pause.click(pause, None, [out, s_start, s_width])
    s_start.change(analyze, [s_start, s_width], [plot, out])
    s_width.change(analyze, [s_start, s_width], [plot, out])

generate_mixed_chunk(0.1)
if __name__ == "__main__": demo.queue().launch(share=True)

# Freq+Mod (OLD)


In [ ]:
# 1. Install Gradio
!pip install gradio -q

import gradio as gr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import time
from collections import Counter



# --- 1. CONFIGURATION ---
class SystemState:
    def __init__(self):
        self.fs = 8000
        self.chunk_duration = 4.0 #keep this a multiple of hop duration
        self.running = False
        self.current_waveform = np.array([])
        self.gt_iq=[]
        self.gt_mod=[]
        self.gt_fc=[]
        self.demod=[]

        # PROFILES (Now with BPSK!)
        self.profiles = {
            "S1 (Long Range)": {
                "freqs": [50, 100],
                "mod": "QPSK",
                "color": "#00CCFF", # Cyan/Blue
                "desc": "Robust / Low Rate"
            },
            "S2 (Standard)": {
                "freqs": [150, 200],
                "mod": "QPSK",
                "color": "#00FF00", # Green
                "desc": "Standard Link"
            },
            "S3 (THREAT)": {
                "freqs": [250, 300, 350],
                "mod": "16-QAM",
                "color": "#FF0000", # Red
                "desc": "High Speed Burst"
            }
        }

sys_state = SystemState()

# --- CLASSFIFER-----
def classify_normalized(i_val, q_val):
    point = i_val + 1j*q_val

    # --- Define NORMALIZED Constellations (Avg Power = 1.0) ---
    # BPSK: +/- 1
    # Power = 1^2 = 1.
    bpsk_pts = [-1, 1]

    # QPSK: (+/- 1 +/- 1j) / sqrt(2)
    # Mag = sqrt(1^2 + 1^2) = sqrt(2). Divided by sqrt(2) = 1.
    scale_qpsk = 1.0 / np.sqrt(2)
    qpsk_pts = [scale_qpsk*(-1-1j), scale_qpsk*(-1+1j), scale_qpsk*(1-1j), scale_qpsk*(1+1j)]

    # 16-QAM: (+/-1, +/-3) / sqrt(10)
    # Avg Power of unscaled 16-QAM is 10. So we divide by sqrt(10).
    scale_16qam = 1.0 / np.sqrt(10)
    vals = [-3, -1, 1, 3]
    qam_pts = [scale_16qam*(x + 1j*y) for x in vals for y in vals]

    # --- Euclidean Distances ---
    err_bpsk = min([abs(point - p) for p in bpsk_pts])
    err_qpsk = min([abs(point - p) for p in qpsk_pts])
    err_16qam = min([abs(point - p) for p in qam_pts])

    # --- Penalties ---
    # Since 16-QAM points are now VERY close together (dense),
    # the penalty is crucial to avoid false positives from noise.
    scores = {
        'BPSK': err_bpsk + 0.0,
        'QPSK': err_qpsk + 0.1,
        '16-QAM': err_16qam + 0.25
    }

    return min(scores, key=scores.get)

# --- 2. MIXED SIGNAL GENERATOR ---
def generate_mixed_chunk(noise_level):
    fs = sys_state.fs
    total_dur = sys_state.chunk_duration

    full_wave = []
    gt_iq=[]
    gt_mod=[]
    gt_fc=[]
    demod=[]
    current_t = 0

    # Generate segments until buffer is full
    while current_t < total_dur:
        # seg_dur = np.random.uniform(1.0, 2.0)
        hop_dur = 0.2
        seg_dur=np.random.randint(0, 5)*hop_dur+1.0
        if current_t + seg_dur > total_dur: seg_dur = total_dur - current_t

        # Pick Random Profile
        p_name = np.random.choice(list(sys_state.profiles.keys()))
        profile = sys_state.profiles[p_name]

        freq_list = profile["freqs"]
        mod_type = profile["mod"]



        # Generate Hopping

        num_hops = int(np.ceil(seg_dur / hop_dur))
        segment_wave = []

        for i in range(num_hops):
            this_hop_dur =hop_dur# min(hop_dur, seg_dur - (i*hop_dur))
            if this_hop_dur <= 0: break

            f_c = freq_list[i % len(freq_list)]
            t = np.linspace(0, this_hop_dur, int(fs*this_hop_dur), endpoint=False)

            # --- MODULATION LOGIC ---
            num_syms = 5#max(5, int(20 * this_hop_dur / 0.2)) # This always 20

            samples_per_sym=int(hop_dur*fs/20)

            if mod_type == "16-QAM":
                scale = 1.0/np.sqrt(10)
                levs = [-3, -1, 1, 3]
                syms = [complex(x,y)*scale for x,y in zip(
                    np.random.choice(levs, num_syms), np.random.choice(levs, num_syms))]

            elif mod_type == "QPSK":
                scale = 1.0/np.sqrt(2)
                syms = [(np.random.choice([-1,1]) + 1j*np.random.choice([-1,1]))*scale
                        for _ in range(num_syms)]

            else: # BPSK (New!)
                # BPSK is just +1 or -1 on the Real axis.
                # We often leave Imaginary as 0.
                syms = [(2*np.random.randint(0,2)-1) + 0j for _ in range(num_syms)]

            # Upsample & Upconvert
            gt_iq=gt_iq+syms
            gt_fc=gt_fc+[f_c]*len(syms)
            gt_mod=gt_mod+[mod_type]*len(syms)

            s_up = np.repeat(syms, int(len(t)/num_syms) + 1)[:len(t)]
            wave = s_up.real * np.cos(2*np.pi*f_c*t) - s_up.imag * np.sin(2*np.pi*f_c*t)

            # print(len(wave),num_syms)
            demod_i=wave*2*np.cos(2*np.pi*f_c*t)
            demod_q=-wave*2*np.sin(2*np.pi*f_c*t)

            demod_i=np.array(np.array_split(demod_i,num_syms))
            demod_q=np.array(np.array_split(demod_q,num_syms))

            # print(demod_i)
            # print(1/0)

            demod_i=np.mean(demod_i,axis=1)
            demod_q=np.mean(demod_q,axis=1)

            demod.append(demod_i+1j*demod_q)
            # print(np.mean(np.abs(demod[-1]-syms)))

            segment_wave.append(wave)

        if segment_wave:
          full_wave.append(np.concatenate(segment_wave))

        current_t += seg_dur

    raw = np.concatenate(full_wave)
    raw = raw[:int(fs*total_dur)]

    sys_state.demod=np.concatenate(demod)

    # Add Noise
    sys_state.current_waveform = raw + (np.random.randn(len(raw)) * noise_level)
    sys_state.gt_iq=gt_iq
    sys_state.gt_fc=gt_fc
    sys_state.gt_mod=gt_mod
# --- 3. DECODER ---
def analyze_selection(start_t, width_t):
    fs = sys_state.fs
    wave = sys_state.current_waveform

    idx_start = int(start_t * fs)
    idx_end = int((start_t + width_t) * fs)
    idx_start = max(0, min(idx_start, len(wave)-1))
    idx_end = max(0, min(idx_end, len(wave)-1))



    if idx_end - idx_start < 50: return None

    chunk = wave[idx_start:idx_end]
     #10 samples per symbol
    sps=int(0.2*fs/5)
    gt_iq=sys_state.gt_iq[int(idx_start/sps):int(idx_end/sps)]
    gt_fc=sys_state.gt_fc[int(idx_start/sps):int(idx_end/sps)]
    gt_mod=sys_state.gt_mod[int(idx_start/sps):int(idx_end/sps)]
    demod=sys_state.demod[int(idx_start/sps):int(idx_end/sps)]

    # 1. Frequency (FFT)
    fft_vals = np.abs(np.fft.fft(chunk))
    fft_freqs = np.fft.fftfreq(len(chunk), 1/fs)
    pos = fft_freqs > 0

    peak_idx = np.argmax(fft_vals[pos])
    dom_freq = fft_freqs[pos][peak_idx]

    threshold = np.max(fft_vals[pos]) * 0.4
    peaks = fft_freqs[pos][fft_vals[pos] > threshold]
    detected_freqs = sorted(list(set([int(round(f/10)*10) for f in peaks if f > 10])))

    # 2. Match Profile
    best_match = "Unknown"
    best_score = 0
    match_color = "gray"
    best_freq_list=[]

    for name, profile in sys_state.profiles.items():
        common = len(set(profile["freqs"]).intersection(set(detected_freqs)))
        if common > best_score:
            best_score = common
            best_match = name
            match_color = profile["color"]
            best_freq_list=profile['freqs']

    print(best_freq_list)

    # Modulation Classification
    samples_per_sym=int(fs*0.2/5) # fs*hop_dur/max_symbols
    num_syms_in_chunk=int(len(chunk)/samples_per_sym)
    sym_chunks=np.array_split(chunk,num_syms_in_chunk)
    t_sym = np.linspace(0, 0.04, samples_per_sym, endpoint=False)

    pred_list=[]
    freq_hop_seq=[]
    rec_constellation=[]
    for i,sym_chunk in enumerate(sym_chunks):
      best_val=-np.inf
      for fc in best_freq_list:
        rec_i = np.mean(sym_chunk * 2 * np.cos(2*np.pi*fc*t_sym))
        rec_q = np.mean(sym_chunk * -2 * np.sin(2*np.pi*fc*t_sym))

        Val=rec_i**2+rec_q**2
        # print(fc,rec_i,rec_q,Val,gt_iq[i])
        if Val>best_val:
          best_val=Val
          best_freq=fc
          best_i=rec_i
          best_q=rec_q
      freq_hop_seq.append(best_freq)
      pred_mod = classify_normalized(best_i, best_q)
      pred_list.append(pred_mod)
      rec_constellation.append(best_i+1j*best_q)
    print('fc',gt_fc)
    print('freq_hop_seq',freq_hop_seq)
    max_vote=Counter(pred_list).most_common(1)
    max_mod=max_vote[0]
    print(pred_list)
    print(max_mod)
    print('GT mod',Counter(gt_mod).most_common(1))

    # 3. Demodulation
    # t_chunk = np.linspace(0, width_t, len(chunk), endpoint=False)
    # baseband = chunk * np.exp(-1j * 2 * np.pi * dom_freq * t_chunk)
    # constellation = baseband[::45] * 2.0 # Scale for visibility

    # Auto-Rotate Phase Correction (Simple)
    # BPSK/QPSK often come out rotated due to phase offset.
    # We rotate the whole cloud so the mean angle aligns with 45 deg or 0 deg for cleaner viewing
    # avg_angle = np.angle(np.mean(constellation**4))/4 # 4th power carrier recovery trick
    # constellation = constellation * np.exp(-1j * avg_angle)

    constellation=np.array(rec_constellation)
    return {
        "seq": best_match,
        "color": match_color,
        "iq": constellation,
        "freqs": detected_freqs,
        "dom_freq": dom_freq,
        "mod": max_mod,
        "freq_hop":best_freq_list
    }

# --- 4. VISUALIZATION ---
def update_plot(scan_start=None, scan_width=None):
    if len(sys_state.current_waveform) == 0: generate_mixed_chunk(0.1)

    wave = sys_state.current_waveform
    fs = sys_state.fs

    fig = plt.figure(figsize=(14, 8))
    fig.patch.set_facecolor('#121212')
    plt.style.use('dark_background')

    gs = fig.add_gridspec(2, 2, width_ratios=[2, 1])
    ax_time = fig.add_subplot(gs[0, 0])
    ax_spec = fig.add_subplot(gs[1, 0])
    ax_iq = fig.add_subplot(gs[:, 1])

    # Standard Plots
    ax_time.plot(np.linspace(0, 4, len(wave)), wave, color='#00FF00', lw=0.5)
    ax_time.set_title("Time Domain", color='white')
    ax_time.set_ylim(-4, 4); ax_time.set_xlim(0, 4)

    ax_spec.specgram(wave, NFFT=256, Fs=fs, noverlap=128, cmap='inferno')
    ax_spec.set_title("Spectrogram", color='white')
    ax_spec.set_ylim(0, 300)

    # IQ Default
    ax_iq.set_title("Constellation Scope", color='white')
    ax_iq.set_xlim(-2.5, 2.5); ax_iq.set_ylim(-2.5, 2.5)
    ax_iq.grid(True, alpha=0.2)
    ax_iq.axhline(0, color='gray'); ax_iq.axvline(0, color='gray')
    ax_iq.text(0,0, "PAUSE & SELECT", ha='center', color='gray')

    status_html = "<h3>Scanning...</h3>"

    if scan_start is not None:
        # Highlight Box
        rect_s = patches.Rectangle((scan_start, -100), scan_width, 300, edgecolor='white', facecolor='yellow', alpha=0.5)
        ax_time.add_patch(rect_s)

        res = analyze_selection(scan_start, scan_width)
        if res:
            ax_iq.clear()
            iq = res["iq"]
            col = res["color"]

            # Scatter
            ax_iq.scatter(iq.real, iq.imag, color=col, s=60, edgecolors='white', alpha=0.9)

            # Guides
            seq = res['mod']#res["seq"]
            if "BPSK" in seq or "Long Range" in seq:
                # BPSK: 2 dots on X axis
                ax_iq.add_patch(patches.Circle((-1,0), 0.3, fill=False, edgecolor='cyan', linestyle='--'))
                ax_iq.add_patch(patches.Circle((1,0), 0.3, fill=False, edgecolor='cyan', linestyle='--'))
                mod_name = "BPSK (2-Point)"
            elif "QAM" in seq:
                mod_name = "16-QAM (16-Point)"
            else:
                mod_name = "QPSK (4-Point)"

            ax_iq.set_title(f"Decoded: {seq}", color=col, fontsize=14, weight='bold')
            ax_iq.set_xlim(-2.5, 2.5); ax_iq.set_ylim(-2.5, 2.5)
            ax_iq.grid(True, alpha=0.2)

            status_html = f"<h3 style='color:{col}'>{res['freq_hop']}</h3>Detected Mod: {seq}"

    plt.tight_layout()
    return fig, status_html

# --- INTERFACE ---
def stream(noise):
    sys_state.running = True
    while sys_state.running:
        generate_mixed_chunk(noise)
        fig, _ = update_plot(None, None)
        yield fig, "<h3>Monitoring...</h3>", gr.update(visible=False), gr.update(visible=False)
        plt.close(fig); time.sleep(2.0)

def pause():
    sys_state.running = False
    return "<h3>Paused</h3>", gr.update(visible=True), gr.update(visible=True)

def analyze(s, w): return update_plot(s, w)

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📡 RF Anomaly Detection Dashboard")
    gr.Markdown("Detect **BPSK (Blue)**, **QPSK (Green)**, and **16-QAM (Red/Threat)**.")
    with gr.Row():
        with gr.Column(scale=1):
            noise = gr.Slider(0, 0.5, 0.1, label="Noise")
            btn_play = gr.Button("▶ Start", variant="primary")
            btn_pause = gr.Button("⏸ Pause", variant="secondary")
            out = gr.HTML("Ready")
            s_start = gr.Slider(0, 3.6, 0.0, step=0.2,label="Time", visible=False)
            s_width = gr.Slider(0.2, 1.6, 0.4,step=0.2, label="Width", visible=False)
        with gr.Column(scale=3): plot = gr.Plot()

    btn_play.click(stream, [noise], [plot, out, s_start, s_width])
    btn_pause.click(pause, None, [out, s_start, s_width])
    s_start.change(analyze, [s_start, s_width], [plot, out])
    s_width.change(analyze, [s_start, s_width], [plot, out])

generate_mixed_chunk(0.1)
# if __name__ == "__main__": demo.queue().launch(share=True)
if __name__ == "__main__": demo.queue().launch(debug=True)

# LLM+Freq_Hop+Mod


In [ ]:
!pip install gradio -q
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q bitsandbytes accelerate gradio

In [ ]:
# 1. Install Gradio

import torch
import gc
from transformers import LlavaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import os

import gradio as gr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import time
from collections import Counter

from PIL import Image


os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
torch.cuda.empty_cache()


univ_hop_dur=0.8
univ_sph=8
# --- 1. CONFIGURATION ---
class SystemState:
    def __init__(self):
        self.fs = 4000
        self.chunk_duration = 4.0 #keep this a multiple of hop duration
        self.running = False
        self.current_waveform = np.array([])
        self.gt_iq=[]
        self.gt_mod=[]
        self.gt_fc=[]
        self.demod=[]
        self.cur_analysis=None

        # PROFILES (Now with BPSK!)
        self.profiles = {
            "S1 (Long Range)": {
                "freqs": [10, 20],
                "mod": "BPSK",
                "color": "#00CCFF", # Cyan/Blue
                "desc": "Robust / Low Rate"
            },
            "S2 (Standard)": {
                "freqs": [30, 40],
                "mod": "QPSK",
                "color": "#00FF00", # Green
                "desc": "Standard Link"
            },
            "S3 (THREAT)": {
                "freqs": [50, 60, 70],
                "mod": "16-QAM",
                "color": "#FF0000", # Red
                "desc": "High Speed Burst"
            }
        }

sys_state = SystemState()

# --- CLASSFIFER-----
def classify_normalized(i_val, q_val):
    point = i_val + 1j*q_val

    # --- Define NORMALIZED Constellations (Avg Power = 1.0) ---
    # BPSK: +/- 1
    # Power = 1^2 = 1.
    bpsk_pts = [-1, 1]

    # QPSK: (+/- 1 +/- 1j) / sqrt(2)
    # Mag = sqrt(1^2 + 1^2) = sqrt(2). Divided by sqrt(2) = 1.
    scale_qpsk = 1.0 / np.sqrt(2)
    qpsk_pts = [scale_qpsk*(-1-1j), scale_qpsk*(-1+1j), scale_qpsk*(1-1j), scale_qpsk*(1+1j)]

    # 16-QAM: (+/-1, +/-3) / sqrt(10)
    # Avg Power of unscaled 16-QAM is 10. So we divide by sqrt(10).
    scale_16qam = 1.0 / np.sqrt(10)
    vals = [-3, -1, 1, 3]
    qam_pts = [scale_16qam*(x + 1j*y) for x in vals for y in vals]

    # --- Euclidean Distances ---
    err_bpsk = min([abs(point - p) for p in bpsk_pts])
    err_qpsk = min([abs(point - p) for p in qpsk_pts])
    err_16qam = min([abs(point - p) for p in qam_pts])

    # --- Penalties ---
    # Since 16-QAM points are now VERY close together (dense),
    # the penalty is crucial to avoid false positives from noise.
    scores = {
        'BPSK': err_bpsk + 0.0,
        'QPSK': err_qpsk + 0.1,
        '16-QAM': err_16qam + 0.25
    }

    return min(scores, key=scores.get)

# --- 2. MIXED SIGNAL GENERATOR ---
def generate_mixed_chunk(noise_level):
    fs = sys_state.fs
    total_dur = sys_state.chunk_duration

    full_wave = []
    gt_iq=[]
    gt_mod=[]
    gt_fc=[]
    demod=[]
    current_t = 0

    # Generate segments until buffer is full
    while current_t < total_dur:
        # seg_dur = np.random.uniform(1.0, 2.0)
        hop_dur = univ_hop_dur
        seg_dur=np.random.randint(0, 5)*hop_dur+1.0
        if current_t + seg_dur > total_dur: seg_dur = total_dur - current_t

        # Pick Random Profile
        p_name = np.random.choice(list(sys_state.profiles.keys()))
        profile = sys_state.profiles[p_name]

        freq_list = profile["freqs"]
        mod_type = profile["mod"]



        # Generate Hopping

        num_hops = int(np.ceil(seg_dur / hop_dur))
        segment_wave = []

        for i in range(num_hops):
            this_hop_dur =hop_dur# min(hop_dur, seg_dur - (i*hop_dur))
            if this_hop_dur <= 0: break

            f_c = freq_list[i % len(freq_list)]
            t = np.linspace(0, this_hop_dur, int(fs*this_hop_dur), endpoint=False)

            # --- MODULATION LOGIC ---
            num_syms = univ_sph#max(5, int(20 * this_hop_dur / 0.2)) # This always 20

            samples_per_sym=int(hop_dur*fs/univ_sph)

            if mod_type == "16-QAM":
                scale = 1.0/np.sqrt(10)
                levs = [-3, -1, 1, 3]
                syms = [complex(x,y)*scale for x,y in zip(
                    np.random.choice(levs, num_syms), np.random.choice(levs, num_syms))]

            elif mod_type == "QPSK":
                scale = 1.0/np.sqrt(2)
                syms = [(np.random.choice([-1,1]) + 1j*np.random.choice([-1,1]))*scale
                        for _ in range(num_syms)]

            else: # BPSK (New!)
                # BPSK is just +1 or -1 on the Real axis.
                # We often leave Imaginary as 0.
                syms = [(2*np.random.randint(0,2)-1) + 0j for _ in range(num_syms)]

            # Upsample & Upconvert
            gt_iq=gt_iq+syms
            gt_fc=gt_fc+[f_c]*len(syms)
            gt_mod=gt_mod+[mod_type]*len(syms)

            s_up = np.repeat(syms, int(len(t)/num_syms) + 1)[:len(t)]
            wave = s_up.real * np.cos(2*np.pi*f_c*t) - s_up.imag * np.sin(2*np.pi*f_c*t)

            # print(len(wave),num_syms)
            demod_i=wave*2*np.cos(2*np.pi*f_c*t)
            demod_q=-wave*2*np.sin(2*np.pi*f_c*t)

            demod_i=np.array(np.array_split(demod_i,num_syms))
            demod_q=np.array(np.array_split(demod_q,num_syms))

            # print(demod_i)
            # print(1/0)

            demod_i=np.mean(demod_i,axis=1)
            demod_q=np.mean(demod_q,axis=1)

            demod.append(demod_i+1j*demod_q)
            # print(np.mean(np.abs(demod[-1]-syms)))

            segment_wave.append(wave)

        if segment_wave:
          full_wave.append(np.concatenate(segment_wave))

        current_t += seg_dur

    raw = np.concatenate(full_wave)
    raw = raw[:int(fs*total_dur)]

    sys_state.demod=np.concatenate(demod)

    # Add Noise
    sys_state.current_waveform = raw + (np.random.randn(len(raw)) * noise_level)
    sys_state.gt_iq=gt_iq
    sys_state.gt_fc=gt_fc
    sys_state.gt_mod=gt_mod
# --- 3. DECODER ---
def analyze_selection(start_t, width_t):
    fs = sys_state.fs
    wave = sys_state.current_waveform

    idx_start = int(start_t * fs)
    idx_end = int((start_t + width_t) * fs)
    idx_start = max(0, min(idx_start, len(wave)-1))
    idx_end = max(0, min(idx_end, len(wave)-1))



    if idx_end - idx_start < 50: return None

    chunk = wave[idx_start:idx_end]
     #10 samples per symbol
    sps=int(univ_hop_dur*fs/univ_sph)
    gt_iq=sys_state.gt_iq[int(idx_start/sps):int(idx_end/sps)]
    gt_fc=sys_state.gt_fc[int(idx_start/sps):int(idx_end/sps)]
    gt_mod=sys_state.gt_mod[int(idx_start/sps):int(idx_end/sps)]
    demod=sys_state.demod[int(idx_start/sps):int(idx_end/sps)]

    # 1. Frequency (FFT)
    fft_vals = np.abs(np.fft.fft(chunk))
    fft_freqs = np.fft.fftfreq(len(chunk), 1/fs)
    pos = fft_freqs > 0

    peak_idx = np.argmax(fft_vals[pos])
    dom_freq = fft_freqs[pos][peak_idx]

    threshold = np.max(fft_vals[pos]) * 0.4
    peaks = fft_freqs[pos][fft_vals[pos] > threshold]
    detected_freqs = sorted(list(set([int(round(f/10)*10) for f in peaks if f > 10])))

    # 2. Match Profile
    best_match = "Unknown"
    best_score = 0
    match_color = "gray"
    best_freq_list=[]

    for name, profile in sys_state.profiles.items():
        common = len(set(profile["freqs"]).intersection(set(detected_freqs)))
        if common > best_score:
            best_score = common
            best_match = name
            match_color = profile["color"]
            best_freq_list=profile['freqs']

    print(best_freq_list)

    # Modulation Classification
    samples_per_sym=int(fs*univ_hop_dur/univ_sph) # fs*hop_dur/max_symbols
    num_syms_in_chunk=int(len(chunk)/samples_per_sym)
    sym_chunks=np.array_split(chunk,num_syms_in_chunk)
    t_sym = np.linspace(0, univ_hop_dur/univ_sph, samples_per_sym, endpoint=False)

    pred_list=[]
    freq_hop_seq=[]
    rec_constellation=[]
    for i,sym_chunk in enumerate(sym_chunks):
      best_val=-np.inf
      for fc in best_freq_list:
        rec_i = np.mean(sym_chunk * 2 * np.cos(2*np.pi*fc*t_sym))
        rec_q = np.mean(sym_chunk * -2 * np.sin(2*np.pi*fc*t_sym))

        Val=rec_i**2+rec_q**2
        # print(fc,rec_i,rec_q,Val,gt_iq[i])
        if Val>best_val:
          best_val=Val
          best_freq=fc
          best_i=rec_i
          best_q=rec_q
      freq_hop_seq.append(best_freq)
      pred_mod = classify_normalized(best_i, best_q)
      pred_list.append(pred_mod)
      rec_constellation.append(best_i+1j*best_q)
    print('fc',gt_fc)
    print('freq_hop_seq',freq_hop_seq)
    max_vote=Counter(pred_list).most_common(1)
    max_mod=max_vote[0]
    print(pred_list)
    print(max_mod)
    print('GT mod',Counter(gt_mod).most_common(1))

    # 3. Demodulation
    # t_chunk = np.linspace(0, width_t, len(chunk), endpoint=False)
    # baseband = chunk * np.exp(-1j * 2 * np.pi * dom_freq * t_chunk)
    # constellation = baseband[::45] * 2.0 # Scale for visibility

    # Auto-Rotate Phase Correction (Simple)
    # BPSK/QPSK often come out rotated due to phase offset.
    # We rotate the whole cloud so the mean angle aligns with 45 deg or 0 deg for cleaner viewing
    # avg_angle = np.angle(np.mean(constellation**4))/4 # 4th power carrier recovery trick
    # constellation = constellation * np.exp(-1j * avg_angle)

    constellation=np.array(rec_constellation)
    return {
        "seq": best_match,
        "color": match_color,
        "iq": constellation,
        "freqs": detected_freqs,
        "dom_freq": dom_freq,
        "mod": max_mod,
        "freq_hop":best_freq_list
    }

# --- 4. VISUALIZATION ---
def update_plot(scan_start=None, scan_width=None):
    if len(sys_state.current_waveform) == 0: generate_mixed_chunk(0.1)

    wave = sys_state.current_waveform
    fs = sys_state.fs

    fig = plt.figure(figsize=(10, 4))
    fig.patch.set_facecolor('#121212')
    plt.style.use('dark_background')

    ax_time = fig.add_subplot(111)


    # gs = fig.add_gridspec(2, 2, width_ratios=[2, 1])
    # ax_time = fig.add_subplot(gs[0, 0])
    # ax_spec = fig.add_subplot(gs[1, 0])
    # ax_iq = fig.add_subplot(gs[:, 1])

    # Standard Plots
    ax_time.plot(np.linspace(0, 4, len(wave)), wave, color='#00FF00', lw=0.5)
    ax_time.set_title("Time Domain", color='white')
    ax_time.set_ylim(-4, 4); ax_time.set_xlim(0, 4)

    # ax_spec.specgram(wave, NFFT=256, Fs=fs, noverlap=128, cmap='inferno')
    # ax_spec.set_title("Spectrogram", color='white')
    # ax_spec.set_ylim(0, 300)

    # IQ Default
    # ax_iq.set_title("Constellation Scope", color='white')
    # ax_iq.set_xlim(-2.5, 2.5); ax_iq.set_ylim(-2.5, 2.5)
    # ax_iq.grid(True, alpha=0.2)
    # ax_iq.axhline(0, color='gray'); ax_iq.axvline(0, color='gray')
    # ax_iq.text(0,0, "PAUSE & SELECT", ha='center', color='gray')

    status_html = "<h3>Scanning...</h3>"

    if scan_start is not None:
        # Highlight Box
        rect_s = patches.Rectangle((scan_start, -100), scan_width, 300, edgecolor='white', facecolor='yellow', alpha=0.5)
        ax_time.add_patch(rect_s)

        # res = analyze_selection(scan_start, scan_width)
    #     if res:
    #         # ax_iq.clear()
    #         iq = res["iq"]
    #         col = res["color"]

    #         # Scatter
    #         ax_iq.scatter(iq.real, iq.imag, color=col, s=60, edgecolors='white', alpha=0.9)

    #         # Guides
    #         seq = res['mod']#res["seq"]
    #         if "BPSK" in seq or "Long Range" in seq:
    #             # BPSK: 2 dots on X axis
    #             ax_iq.add_patch(patches.Circle((-1,0), 0.3, fill=False, edgecolor='cyan', linestyle='--'))
    #             ax_iq.add_patch(patches.Circle((1,0), 0.3, fill=False, edgecolor='cyan', linestyle='--'))
    #             mod_name = "BPSK (2-Point)"
    #         elif "QAM" in seq:
    #             mod_name = "16-QAM (16-Point)"
    #         else:
    #             mod_name = "QPSK (4-Point)"

    #         ax_iq.set_title(f"Decoded: {seq}", color=col, fontsize=14, weight='bold')
    #         ax_iq.set_xlim(-2.5, 2.5); ax_iq.set_ylim(-2.5, 2.5)
    #         ax_iq.grid(True, alpha=0.2)

    #         status_html = f"<h3 style='color:{col}'>{res['freq_hop']}</h3>Detected Mod: {seq}"

    # plt.tight_layout()
    return fig, status_html

# ---- CHATBOT -----

# --- CONFIGURATION ---
MODEL_ID = "llava-hf/llava-1.5-7b-hf"

# --- SYSTEM PROMPT ---
SYSTEM_PROMPT = (
    "You are an RF engineering in the US army."
    "Ignore any images given."
    "You have access to a RF tool which can analyze and provide the frequency hopping sequence and the modulation scheme that is observed currently."
    "Currently the US army employs the following two modulation and frequency hopping sequences."
    "That is a legitate and legal RF signal would either have a frequency hopping sequence of [10,20] andd would use BPSK modulation or it would have a frequency hopping sequence of [30,40] with QPSK modulation."
    "If the modulation scheme is not QPSK or BPSK or if the frequency hopping sequence is not one of [10,20] or [30,40], you must warn the operator immediately."
    "The outputs from the RF tool will be given between START_RFT and END_RFT words."
    "Based on the results from this RF tool, you task is to answer the questions so that even a civilian would understand what is going on."
)

print(f"Loading {MODEL_ID}...")
# --- LOAD MODEL ---
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)
print(type(MODEL_ID))
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto"
)

print("Model loaded successfully!")

def format_prompt(history, new_message):
    prompt = "USER: <image>\n"
    if len(history) == 0:
        prompt += f"{SYSTEM_PROMPT}\n\n"
    for user_msg, bot_msg in history:
        prompt += f"{user_msg}\nASSISTANT: {bot_msg}\nUSER: "
    #print(sys_state.cur_analysis['mod'],str(sys_state.cur_analysis['freq_hop']))
    rft_prompt="START_RFT Detected modulation is "+sys_state.cur_analysis['mod'][0]+". Detected frequency hopping sequence is "+str(sys_state.cur_analysis['freq_hop'])+ ". END_RFT"
    prompt += f"{new_message}\n{rft_prompt}\nASSISTANT:"
    return prompt

def chat_fn(message, history):
    if not message: return "Please enter a question."
    # if not image_state: return "Please upload an image."

    try:
        processed_image = Image.new("RGB", (3, 3), color = 'white')#None#crop_image_to_mask(image_state)
        full_prompt = format_prompt(history, message)

        inputs = processor(text=full_prompt, images=processed_image, return_tensors="pt").to("cuda")

        with torch.backends.cuda.sdp_kernel(enable_flash=True, enable_math=False, enable_mem_efficient=True):
            generate_ids = model.generate(
                **inputs,
                max_new_tokens=300,
                do_sample=True,
                temperature=0.5,
                top_p=0.9
            )

        response = processor.batch_decode(generate_ids, skip_special_tokens=True)[0]
        #response=[]

        if "ASSISTANT:" in response:
            final_response = response.split("ASSISTANT:")[-1].strip()
        else:
            final_response = response

        torch.cuda.empty_cache()
        return final_response

    except Exception as e:
        torch.cuda.empty_cache()
        return f"Error: {str(e)}"

def respond(message, chat_history):

        bot_message = chat_fn(message, chat_history)
        chat_history.append((message, bot_message))
        return "", chat_history

# --- INTERFACE ---
def stream(noise):
    sys_state.running = True
    while sys_state.running:
        generate_mixed_chunk(noise)
        fig, _ = update_plot(None, None)
        yield fig, "<h3>Monitoring...</h3>", gr.update(visible=True), gr.update(visible=True)
        plt.close(fig); time.sleep(2.0)

def pause(s,w):
    sys_state.running = False
    res = analyze_selection(s, w)
    sys_state.cur_analysis=res
    pause_fig,_=update_plot(s,w)
    return "<h3>Paused</h3>", gr.update(visible=True), gr.update(visible=True),pause_fig

def analyze(s, w):
  res = analyze_selection(s, w)
  sys_state.cur_analysis=res
  return update_plot(s, w)

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📡 3-Modulation Interceptor")
    gr.Markdown("Detect **BPSK (Blue)**, **QPSK (Green)**, and **16-QAM (Red/Threat)**.")
    with gr.Row():
        with gr.Column(scale=3):
          with gr.Row():
            plot = gr.Plot()
          with gr.Row():
            with gr.Column(scale=0.3):
              noise = gr.Slider(0, 0.5, 0.1, label="Noise")
              btn_play = gr.Button("▶ Start", variant="primary")
              btn_pause = gr.Button("⏸ Pause", variant="secondary")
              out = gr.HTML("Ready")
              s_start = gr.Slider(0, sys_state.chunk_duration-2*univ_hop_dur, 0.0, step=univ_hop_dur,label="Time", visible=True)
              s_width = gr.Slider(univ_hop_dur, 2*univ_hop_dur, univ_hop_dur,step=univ_hop_dur, label="Width", visible=True)
            with gr.Column(scale=1):
              plot2 = gr.Plot()
        with gr.Column(scale=1):
          chatbot = gr.Chatbot(label="Skynet", height=500)
          msg = gr.Textbox(label="Your Message")
          clear = gr.Button("Clear Memory")

    btn_play.click(stream, [noise], [plot, out, s_start, s_width])
    btn_pause.click(pause, [s_start, s_width], [out, s_start, s_width, plot])
    s_start.change(analyze, [s_start, s_width], [plot, out])
    s_width.change(analyze, [s_start, s_width], [plot, out])

    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: None, None, chatbot, queue=False)

generate_mixed_chunk(0.1)
# if __name__ == "__main__": demo.queue().launch(share=True)
if __name__ == "__main__": demo.queue().launch(debug=True)

# Drone audio


In [ ]:
!git clone https://github.com/saraalemadi/DroneAudioDataset.git

In [ ]:
%cd DroneAudioDataset

In [ ]:
%%writefile train_cnn.py
import os
import math
import random
import argparse
from dataclasses import dataclass
from typing import List, Tuple, Dict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
# ----------------------------
# Repro
# ----------------------------
def seed_everything(seed: int = 1337) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ----------------------------
# Dataset indexing
# ----------------------------
AUDIO_EXTS = (".wav", ".flac", ".mp3", ".ogg", ".m4a")


def list_class_folders(data_dir: str) -> List[str]:
    classes = []
    for name in sorted(os.listdir(data_dir)):
        p = os.path.join(data_dir, name)
        if os.path.isdir(p) and not name.startswith("."):
            classes.append(name)
    if not classes:
        raise RuntimeError(f"No class folders found in: {data_dir}")
    return classes


def index_files(data_dir: str) -> Tuple[List[str], List[int], List[str], Dict[str, int]]:
    classes = list_class_folders(data_dir)
    class_to_idx = {c: i for i, c in enumerate(classes)}

    paths: List[str] = []
    labels: List[int] = []

    for c in classes:
        cdir = os.path.join(data_dir, c)
        for root, _, files in os.walk(cdir):
            for fn in files:
                if fn.lower().endswith(AUDIO_EXTS):
                    paths.append(os.path.join(root, fn))
                    labels.append(class_to_idx[c])

    if not paths:
        raise RuntimeError(f"No audio files found under: {data_dir}")

    return paths, labels, classes, class_to_idx
# ----------------------------
# Audio -> log-mel + aug
# ----------------------------
@dataclass
class AudioConfig:
    sample_rate: int = 16000
    clip_seconds: float = 2.5  # fixed length clips
    n_fft: int = 1024
    hop_length: int = 256
    n_mels: int = 64
    f_min: int = 30
    f_max: int = 8000
    time_mask_param: int = 24
    freq_mask_param: int = 10


class DroneAudioDataset(Dataset):
    def __init__(
        self,
        paths: List[str],
        labels: List[int],
        cfg: AudioConfig,
        train: bool = True,
    ):
        self.paths = paths
        self.labels = labels
        self.cfg = cfg
        self.train = train

        self.melspec = torchaudio.transforms.MelSpectrogram(
            sample_rate=cfg.sample_rate,
            n_fft=cfg.n_fft,
            hop_length=cfg.hop_length,
            n_mels=cfg.n_mels,
            f_min=cfg.f_min,
            f_max=cfg.f_max,
            power=2.0,
        )
        self.to_db = torchaudio.transforms.AmplitudeToDB(stype="power")

        # SpecAugment-style
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=cfg.time_mask_param)
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=cfg.freq_mask_param)

    def _load_resample_mono(self, path: str) -> torch.Tensor:
        wav, sr = torchaudio.load(path)  # [C, T]
        if wav.size(0) > 1:
            wav = wav.mean(dim=0, keepdim=True)  # mono
        if sr != self.cfg.sample_rate:
            wav = torchaudio.transforms.Resample(sr, self.cfg.sample_rate)(wav)
        return wav  # [1, T]

    def _crop_or_pad(self, wav: torch.Tensor) -> torch.Tensor:
        target_len = int(self.cfg.sample_rate * self.cfg.clip_seconds)
        T = wav.size(1)
        if T == target_len:
            return wav
        if T > target_len:
            # random crop during train, center crop during eval
            if self.train:
                start = random.randint(0, T - target_len)
            else:
                start = (T - target_len) // 2
            return wav[:, start : start + target_len]
        # pad
        pad = target_len - T
        return F.pad(wav, (0, pad), mode="constant", value=0.0)

    def _wav_to_logmel(self, wav: torch.Tensor) -> torch.Tensor:
        # wav: [1, T]
        mel = self.melspec(wav)         # [1, n_mels, frames]
        logmel = self.to_db(mel)        # dB
        # normalize per-sample
        logmel = (logmel - logmel.mean()) / (logmel.std() + 1e-6)
        return logmel                   # [1, n_mels, frames]

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, i: int):
        path = self.paths[i]
        y = self.labels[i]

        wav = self._load_resample_mono(path)
        wav = self._crop_or_pad(wav)

        # light waveform augmentation
        if self.train:
            # random gain
            gain = 10 ** random.uniform(-0.15, 0.15)
            wav = wav * gain
            # tiny gaussian noise
            if random.random() < 0.25:
                wav = wav + 0.003 * torch.randn_like(wav)

        x = self._wav_to_logmel(wav)  # [1, M, F]

        if self.train:
            if random.random() < 0.7:
                x = self.freq_mask(x)
            if random.random() < 0.7:
                x = self.time_mask(x)

        return x, y
# ----------------------------
# CNN model
# ----------------------------
class SmallAudioCNN(nn.Module):
    def __init__(self, n_classes: int):
        super().__init__()
        # input: [B, 1, n_mels, frames]
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.25),
            nn.Linear(128, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.classifier(x)
        return x
# ----------------------------
# Train / eval
# ----------------------------
@torch.no_grad()
def evaluate(model, loader, device) -> Tuple[float, np.ndarray, np.ndarray]:
    model.eval()
    loss_fn = nn.CrossEntropyLoss()

    total_loss = 0.0
    n = 0
    all_y = []
    all_p = []

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)
        loss = loss_fn(logits, y)

        total_loss += loss.item() * x.size(0)
        n += x.size(0)

        preds = torch.argmax(logits, dim=1)
        all_y.append(y.cpu().numpy())
        all_p.append(preds.cpu().numpy())

    return total_loss / max(n, 1), np.concatenate(all_y), np.concatenate(all_p)
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", type=str, default=".", help="Path to Multiclass_Drone_Audio (contains class folders)")
    ap.add_argument("--epochs", type=int, default=25)
    ap.add_argument("--batch_size", type=int, default=32)
    ap.add_argument("--lr", type=float, default=2e-3)
    ap.add_argument("--seed", type=int, default=1337)
    ap.add_argument("--num_workers", type=int, default=2)
    ap.add_argument("--val_frac", type=float, default=0.15)
    ap.add_argument("--test_frac", type=float, default=0.15)
    args = ap.parse_args()

    seed_everything(args.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    paths, labels, classes, _ = index_files(args.data_dir)
    labels_np = np.array(labels)

    # split train/val/test stratified
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=args.test_frac, random_state=args.seed)
    trainval_idx, test_idx = next(sss1.split(np.zeros_like(labels_np), labels_np))

    labels_trainval = labels_np[trainval_idx]
    val_size = args.val_frac / (1.0 - args.test_frac)

    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_size, random_state=args.seed)
    train_idx, val_idx_rel = next(sss2.split(np.zeros_like(labels_trainval), labels_trainval))
    val_idx = trainval_idx[val_idx_rel]
    train_idx = trainval_idx[train_idx]

    def subset(idxs):
        return [paths[i] for i in idxs], [labels[i] for i in idxs]

    train_paths, train_labels = subset(train_idx)
    val_paths, val_labels = subset(val_idx)
    test_paths, test_labels = subset(test_idx)

    cfg = AudioConfig()
    train_ds = DroneAudioDataset(train_paths, train_labels, cfg, train=True)
    val_ds = DroneAudioDataset(val_paths, val_labels, cfg, train=False)
    test_ds = DroneAudioDataset(test_paths, test_labels, cfg, train=False)

    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True,
                              num_workers=args.num_workers, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False,
                            num_workers=args.num_workers, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=args.batch_size, shuffle=False,
                             num_workers=args.num_workers, pin_memory=True)

    model = SmallAudioCNN(n_classes=len(classes)).to(device)

    # handle imbalance (optional but usually helps)
    counts = np.bincount(np.array(train_labels), minlength=len(classes))
    weights = (counts.sum() / (counts + 1e-9))
    weights = weights / weights.mean()
    class_weights = torch.tensor(weights, dtype=torch.float32, device=device)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)

    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs)

    best_val = float("inf")
    best_path = "best_cnn.pt"

    for epoch in range(1, args.epochs + 1):
        model.train()
        running = 0.0
        n = 0

        pbar = tqdm(train_loader, desc=f"epoch {epoch}/{args.epochs}", leave=False)
        for x, y in pbar:
            x = x.to(device)
            y = y.to(device)

            opt.zero_grad(set_to_none=True)
            logits = model(x)
            loss = loss_fn(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()

            running += loss.item() * x.size(0)
            n += x.size(0)
            pbar.set_postfix(loss=running / max(n, 1))

        sched.step()

        val_loss, vy, vp = evaluate(model, val_loader, device)
        val_acc = (vp == vy).mean()

        print(f"[epoch {epoch}] train_loss={running/max(n,1):.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.3f}")

        if val_loss < best_val:
            best_val = val_loss
            torch.save(
                {"model": model.state_dict(), "classes": classes, "cfg": cfg.__dict__},
                best_path,
            )
            print(f"  saved: {best_path}")

    # Test with best checkpoint
    ckpt = torch.load(best_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    test_loss, ty, tp = evaluate(model, test_loader, device)
    test_acc = (tp == ty).mean()

    print("\n=== TEST ===")
    print(f"test_loss={test_loss:.4f}  test_acc={test_acc:.3f}")
    print("\nClasses:", classes)
    print("\nClassification report:\n", classification_report(ty, tp, target_names=classes, digits=3))
    print("\nConfusion matrix:\n", confusion_matrix(ty, tp))

if __name__ == "__main__":
    main()

In [ ]:
!python train_cnn.py --data_dir Multiclass_Drone_Audio --epochs 20 --batch_size 32

In [ ]:
import torch
import torchaudio
import torch.nn as nn
import random
import os
from IPython.display import Audio

device = "cuda" if torch.cuda.is_available() else "cpu"

# CNN architecture (same as training)
class SmallAudioCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16,32,3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten(),
            nn.Dropout(0.25), # Added the missing Dropout layer
            nn.Linear(128,n_classes)
        )

    def forward(self,x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
ckpt = torch.load("best_cnn.pt", map_location=device)

classes = ckpt["classes"]

model = SmallAudioCNN(len(classes)).to(device)
model.load_state_dict(ckpt["model"])
model.eval()

print("Loaded model with classes:", classes)

In [ ]:
dataset_path = "Multiclass_Drone_Audio"

files = []

for c in classes:
    folder = os.path.join(dataset_path, c)
    for f in os.listdir(folder):
        if f.endswith(".wav"):
            files.append(os.path.join(folder, f))

audio_path = random.choice(files)

print("Random sample:", audio_path)

waveform, sr = torchaudio.load(audio_path)
Audio(waveform.numpy(), rate=sr)

In [ ]:
# convert to mono
waveform = waveform.mean(dim=0, keepdim=True)

# resample
waveform = torchaudio.transforms.Resample(sr,16000)(waveform)

mel = torchaudio.transforms.MelSpectrogram(
    sample_rate=16000,
    n_fft=1024,
    hop_length=256,
    n_mels=64
)

to_db = torchaudio.transforms.AmplitudeToDB()

spec = mel(waveform)
spec = to_db(spec)

spec = (spec - spec.mean())/(spec.std()+1e-6)

spec = spec.unsqueeze(0).to(device)
with torch.no_grad():
    logits = model(spec)
    pred = torch.argmax(logits, dim=1).item()

predicted_class = classes[pred]

#print("Predicted class:", predicted_class)

if predicted_class == "unknown":
    print("No drone detected")
else:
    print("Drone detected:", predicted_class)

# Tool Calling + Audio (with embedding only)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
## Dataset Preparation
import tensorflow as tf
class DroneEmbeddingDataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.samples = []
        self.labels = []

        # Binary Categories (ERAU Dataset Structure)
        categories = {'no_drone': 0, 'drone': 1}

        for category, label in categories.items():
            folder_path = os.path.join(root_dir, category)
            if not os.path.exists(folder_path):
                print(f"Skipping {category}: path not found.")
                continue

            # Find all .tfdata files
            for file in os.listdir(folder_path):
                if file.endswith('.tfdata'):
                    self.samples.append(os.path.join(folder_path, file))
                    self.labels.append(label)

        print(f"Successfully loaded {len(self.samples)} samples.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path = self.samples[idx]

        # 1. Read the serialized .tfdata file using TensorFlow
        raw_data = tf.io.read_file(file_path)

        # 2. Parse the tensor (YAMNet embeddings are float32, 1024-dim)
        embedding_tf = tf.io.parse_tensor(raw_data, out_type=tf.float32)

        # 3. Convert to NumPy then to PyTorch tensor and ensure 1024 dimensions
        embedding_np = embedding_tf.numpy().reshape(-1, 1024) # Reshape to (N, 1024)
        embedding_np = embedding_np[0, :]                     # Take the first 1024-dim embedding
        embedding_pt = t.from_numpy(embedding_np).float()

        label = self.labels[idx]
        return embedding_pt, t.tensor(label, dtype=t.float32).view(1)

# Ensure path matches your Drive exactly
dataset_root = '/content/drive/MyDrive/drone_dataset'
dataset = DroneEmbeddingDataset(dataset_root)

In [ ]:
from torch.utils.data import random_split

# 1. Total number of samples in your dataset
total_size = len(dataset)
train_size = int(0.8 * total_size)
test_size = total_size - train_size

# 2. Perform the split
# Using a manual_seed ensures you get the same split every time you run it
train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=t.Generator().manual_seed(42)
)

# 3. Create two separate DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")

In [ ]:
## Binary Classification head
class DroneMLP(nn.Module):
    def __init__(self):
        super(DroneMLP, self).__init__()
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512),  # Input is the 1024 YAMNet embedding
            nn.ReLU(),
            nn.Dropout(0.3),       # Prevents overfitting
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1),     # Binary output (0 or 1)
            nn.Sigmoid()           # Squash to 0.0 - 1.0 probability
        )

    def forward(self, x):
        return self.classifier(x)

# Initialize
device = t.device("cuda" if t.cuda.is_available() else "cpu")
model = DroneMLP().to(device)

In [ ]:
def evaluate(loader):
    model.eval()
    correct = 0
    total = 0
    with t.no_grad():
        for embeddings, labels in loader:
            embeddings, labels = embeddings.to(device), labels.to(device)
            outputs = model(embeddings)

            # Since output is sigmoid (0-1), round to 0 or 1
            predicted = (outputs > 0.5).float()
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total

def train(epochs=10):
    # Re-initialize DroneMLP to ensure the correct model is being trained
    global model # Declare model as global to modify the existing global variable
    model = DroneMLP().to(device)

    criterion = nn.BCELoss()
    optimizer = t.optim.Adam(model.parameters(), lr=0.001)
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for embeddings, labels in train_loader:
            embeddings, labels = embeddings.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(embeddings)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        print(f"Epoch {epoch+1} | Loss: {epoch_loss/len(train_loader):.4f}")

# Start the training
device = "cuda" if t.cuda.is_available() else "cpu"
model.to(device) # Move the model to the correct device
train(epochs=5)
# for epoch in range(15):
#   # Run evaluation on test set
#   test_acc = evaluate(test_loader)
#   print(f"Epoch {epoch+1} | Test Acc: {test_acc:.2f}%")

In [ ]:
!pip install -qU gradio nest-asyncio mcp langchain-openai langchain-mcp-adapters langgraph langchain-ollama

In [ ]:
# !sudo apt-get install -y zstd

# # 1. Install Ollama
# !curl -fsSL https://ollama.com/install.sh | sh

# # 2. Start Ollama server in the background
# import threading
# import subprocess
# import time

# def run_ollama():
#     # 'ollama serve' starts the API server
#     subprocess.run(["ollama", "serve"])

# # Launch the server thread
# ollama_thread = threading.Thread(target=run_ollama, daemon=True)
# ollama_thread.start()

# # Wait a few seconds for the server to wake up
# time.sleep(5)
# print("✅ Ollama server is running!")
# !ollama pull llama3.2

In [ ]:
# 1. Install Gradio (Assuming this is run in a standard environment)
import torch
import gc
import os

import gradio as gr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import time
from collections import Counter

from PIL import Image

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
torch.cuda.empty_cache()

univ_hop_dur=0.8
univ_sph=8

# --- 1. CONFIGURATION ---
class SystemState:
    def __init__(self):
        self.fs = 4000
        self.chunk_duration = 4.0 #keep this a multiple of hop duration
        self.running = False
        self.current_waveform = np.array([])
        self.gt_iq=[]
        self.gt_mod=[]
        self.gt_fc=[]
        self.demod=[]
        self.cur_analysis=None

        # PROFILES (Now with BPSK!)
        self.profiles = {
            "S1 (Long Range)": {
                "freqs": [10, 20],
                "mod": "BPSK",
                "color": "#00CCFF", # Cyan/Blue
                "desc": "Robust / Low Rate"
            },
            "S2 (Standard)": {
                "freqs": [30, 40],
                "mod": "QPSK",
                "color": "#00FF00", # Green
                "desc": "Standard Link"
            },
            "S3 (THREAT)": {
                "freqs": [50, 60, 70],
                "mod": "16-QAM",
                "color": "#FF0000", # Red
                "desc": "High Speed Burst"
            }
        }

sys_state = SystemState()

# --- CLASSFIFER-----
def classify_normalized(i_val, q_val):
    point = i_val + 1j*q_val

    # --- Define NORMALIZED Constellations (Avg Power = 1.0) ---
    bpsk_pts = [-1, 1]

    scale_qpsk = 1.0 / np.sqrt(2)
    qpsk_pts = [scale_qpsk*(-1-1j), scale_qpsk*(-1+1j), scale_qpsk*(1-1j), scale_qpsk*(1+1j)]

    scale_16qam = 1.0 / np.sqrt(10)
    vals = [-3, -1, 1, 3]
    qam_pts = [scale_16qam*(x + 1j*y) for x in vals for y in vals]

    # --- Euclidean Distances ---
    err_bpsk = min([abs(point - p) for p in bpsk_pts])
    err_qpsk = min([abs(point - p) for p in qpsk_pts])
    err_16qam = min([abs(point - p) for p in qam_pts])

    # --- Penalties ---
    scores = {
        'BPSK': err_bpsk + 0.0,
        'QPSK': err_qpsk + 0.1,
        '16-QAM': err_16qam + 0.25
    }

    return min(scores, key=scores.get)

# --- 2. MIXED SIGNAL GENERATOR ---
def generate_mixed_chunk(noise_level):
    fs = sys_state.fs
    total_dur = sys_state.chunk_duration

    full_wave = []
    gt_iq=[]
    gt_mod=[]
    gt_fc=[]
    demod=[]
    current_t = 0

    # Generate segments until buffer is full
    while current_t < total_dur:
        hop_dur = univ_hop_dur
        seg_dur=np.random.randint(0, 5)*hop_dur+1.0
        if current_t + seg_dur > total_dur: seg_dur = total_dur - current_t

        # Pick Random Profile
        p_name = np.random.choice(list(sys_state.profiles.keys()))
        profile = sys_state.profiles[p_name]

        freq_list = profile["freqs"]
        mod_type = profile["mod"]

        # Generate Hopping
        num_hops = int(np.ceil(seg_dur / hop_dur))
        segment_wave = []

        for i in range(num_hops):
            this_hop_dur =hop_dur
            if this_hop_dur <= 0: break

            f_c = freq_list[i % len(freq_list)]
            t = np.linspace(0, this_hop_dur, int(fs*this_hop_dur), endpoint=False)

            # --- MODULATION LOGIC ---
            num_syms = univ_sph

            samples_per_sym=int(hop_dur*fs/univ_sph)

            if mod_type == "16-QAM":
                scale = 1.0/np.sqrt(10)
                levs = [-3, -1, 1, 3]
                syms = [complex(x,y)*scale for x,y in zip(
                    np.random.choice(levs, num_syms), np.random.choice(levs, num_syms))]

            elif mod_type == "QPSK":
                scale = 1.0/np.sqrt(2)
                syms = [(np.random.choice([-1,1]) + 1j*np.random.choice([-1,1]))*scale
                        for _ in range(num_syms)]

            else: # BPSK (New!)
                syms = [(2*np.random.randint(0,2)-1) + 0j for _ in range(num_syms)]

            # Upsample & Upconvert
            gt_iq=gt_iq+syms
            gt_fc=gt_fc+[f_c]*len(syms)
            gt_mod=gt_mod+[mod_type]*len(syms)

            s_up = np.repeat(syms, int(len(t)/num_syms) + 1)[:len(t)]
            wave = s_up.real * np.cos(2*np.pi*f_c*t) - s_up.imag * np.sin(2*np.pi*f_c*t)

            demod_i=wave*2*np.cos(2*np.pi*f_c*t)
            demod_q=-wave*2*np.sin(2*np.pi*f_c*t)

            demod_i=np.array(np.array_split(demod_i,num_syms))
            demod_q=np.array(np.array_split(demod_q,num_syms))

            demod_i=np.mean(demod_i,axis=1)
            demod_q=np.mean(demod_q,axis=1)

            demod.append(demod_i+1j*demod_q)

            segment_wave.append(wave)

        if segment_wave:
          full_wave.append(np.concatenate(segment_wave))

        current_t += seg_dur

    raw = np.concatenate(full_wave)
    raw = raw[:int(fs*total_dur)]

    sys_state.demod=np.concatenate(demod)

    # Add Noise
    sys_state.current_waveform = raw + (np.random.randn(len(raw)) * noise_level)
    sys_state.gt_iq=gt_iq
    sys_state.gt_fc=gt_fc
    sys_state.gt_mod=gt_mod

# --- 3. DECODER ---
def analyze_selection(start_t, width_t):
    fs = sys_state.fs
    wave = sys_state.current_waveform

    idx_start = int(start_t * fs)
    idx_end = int((start_t + width_t) * fs)
    idx_start = max(0, min(idx_start, len(wave)-1))
    idx_end = max(0, min(idx_end, len(wave)-1))

    if idx_end - idx_start < 50: return None

    chunk = wave[idx_start:idx_end]
     #10 samples per symbol
    sps=int(univ_hop_dur*fs/univ_sph)
    gt_iq=sys_state.gt_iq[int(idx_start/sps):int(idx_end/sps)]
    gt_fc=sys_state.gt_fc[int(idx_start/sps):int(idx_end/sps)]
    gt_mod=sys_state.gt_mod[int(idx_start/sps):int(idx_end/sps)]
    demod=sys_state.demod[int(idx_start/sps):int(idx_end/sps)]

    # 1. Frequency (FFT)
    fft_vals = np.abs(np.fft.fft(chunk))
    fft_freqs = np.fft.fftfreq(len(chunk), 1/fs)
    pos = fft_freqs > 0

    peak_idx = np.argmax(fft_vals[pos])
    dom_freq = fft_freqs[pos][peak_idx]

    threshold = np.max(fft_vals[pos]) * 0.4
    peaks = fft_freqs[pos][fft_vals[pos] > threshold]
    detected_freqs = sorted(list(set([int(round(f/10)*10) for f in peaks if f > 10])))

    # 2. Match Profile
    best_match = "Unknown"
    best_score = 0
    match_color = "gray"
    best_freq_list=[]

    for name, profile in sys_state.profiles.items():
        common = len(set(profile["freqs"]).intersection(set(detected_freqs)))
        if common > best_score:
            best_score = common
            best_match = name
            match_color = profile["color"]
            best_freq_list=profile['freqs']

    print(best_freq_list)

    # Modulation Classification
    samples_per_sym=int(fs*univ_hop_dur/univ_sph) # fs*hop_dur/max_symbols
    num_syms_in_chunk=int(len(chunk)/samples_per_sym)
    sym_chunks=np.array_split(chunk,num_syms_in_chunk)
    t_sym = np.linspace(0, univ_hop_dur/univ_sph, samples_per_sym, endpoint=False)

    pred_list=[]
    freq_hop_seq=[]
    rec_constellation=[]
    for i,sym_chunk in enumerate(sym_chunks):
      best_val=-np.inf
      for fc in best_freq_list:
        rec_i = np.mean(sym_chunk * 2 * np.cos(2*np.pi*fc*t_sym))
        rec_q = np.mean(sym_chunk * -2 * np.sin(2*np.pi*fc*t_sym))

        Val=rec_i**2+rec_q**2
        if Val>best_val:
          best_val=Val
          best_freq=fc
          best_i=rec_i
          best_q=rec_q
      freq_hop_seq.append(best_freq)
      pred_mod = classify_normalized(best_i, best_q)
      pred_list.append(pred_mod)
      rec_constellation.append(best_i+1j*best_q)
    print('fc',gt_fc)
    print('freq_hop_seq',freq_hop_seq)
    max_vote=Counter(pred_list).most_common(1)
    max_mod=max_vote[0]
    print(pred_list)
    print(max_mod)
    print('GT mod',Counter(gt_mod).most_common(1))

    constellation=np.array(rec_constellation)
    return {
        "seq": best_match,
        "color": match_color,
        "iq": constellation,
        "freqs": detected_freqs,
        "dom_freq": dom_freq,
        "mod": max_mod,
        "freq_hop":best_freq_list
    }

# --- 4. VISUALIZATION ---
def update_plot(scan_start=None, scan_width=None):
    if len(sys_state.current_waveform) == 0: generate_mixed_chunk(0.1)

    wave = sys_state.current_waveform
    fs = sys_state.fs

    fig = plt.figure(figsize=(10, 4))
    fig.patch.set_facecolor('#121212')
    plt.style.use('dark_background')

    ax_time = fig.add_subplot(111)

    # Standard Plots
    ax_time.plot(np.linspace(0, 4, len(wave)), wave, color='#00FF00', lw=0.5)
    ax_time.set_title("Time Domain", color='white')
    ax_time.set_ylim(-4, 4); ax_time.set_xlim(0, 4)

    status_html = "<h3>Scanning...</h3>"

    if scan_start is not None:
        # Highlight Box
        rect_s = patches.Rectangle((scan_start, -100), scan_width, 300, edgecolor='white', facecolor='yellow', alpha=0.5)
        ax_time.add_patch(rect_s)

    return fig, status_html

# ---- CHATBOT -----

import nest_asyncio
import threading
import asyncio
from mcp.server.fastmcp import FastMCP
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage

# Apply nest_asyncio to allow nested event loops in Colab/Jupyter/Gradio
nest_asyncio.apply()

# --- 1. SET UP FAST MCP SERVER ---
mcp = FastMCP("RF_Analysis_Assistant")

@mcp.tool()
def check_current_rf_status() -> str:
    """Reads the current RF environment variables including modulation and frequency hopping."""
    if sys_state.cur_analysis:
        mod = sys_state.cur_analysis['mod'][0]
        freq_hop = str(sys_state.cur_analysis['freq_hop'])
        return f"START_RFT Detected modulation is {mod}. Detected frequency hopping sequence is {freq_hop}. END_RFT"
    return "No RF analysis available. The operator has not scanned a signal yet."

@mcp.tool()
def check_current_audio_status() -> str:
  """Checks audio signals to detect whether a drone is present."""
  from torch.utils.data import RandomSampler
  time_start = 1
  time_end = 20
  result = []
  num_samples = int(time_end-time_start)+1
  random_sampler = RandomSampler(test_dataset, num_samples=num_samples, replacement=False)
  sampled_dataloader = DataLoader(test_dataset, batch_size=1, sampler=random_sampler)
  model.eval()
  with t.no_grad():
    for embeddings, labels in sampled_dataloader:
      embeddings, labels = embeddings.to(device), labels.to(device)
      output = model(embeddings)
      result.append((output > 0.5).float())
  print(result)
  if any(pred.item() == 1.0 for pred in result):
    return 'The current audio signal was checked and a drone was detected!'
  else:
    return 'The current audio signal was checked and no drone was detected!'

def start_server():
    print("Starting MCP Server on SSE transport...")
    mcp.run(transport="sse")

# Start MCP Server in the background
server_thread = threading.Thread(target=start_server, name="mcp_server_thread", daemon=True)
server_thread.start()

!sudo apt-get install -y zstd

# 1. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Start Ollama server in the background
import threading
import subprocess
import time

def run_ollama():
    # 'ollama serve' starts the API server
    subprocess.run(["ollama", "serve"])

# Launch the server thread
ollama_thread = threading.Thread(target=run_ollama, daemon=True)
ollama_thread.start()

# Wait a few seconds for the server to wake up
time.sleep(5)
print("✅ Ollama server is running!")
!ollama pull llama3.2

# --- 2. SET UP LANGGRAPH & OLLAMA ---
# Initialize the LLM (Ensure 'ollama serve' is running in your environment!)
from langchain_ollama import ChatOllama

# ChatOllama is specifically built to handle Llama 3.2's tool-calling quirks
llm = ChatOllama(
    model="llama3.2",
    temperature=0 # Keep temperature at 0 to make tool calling more deterministic and stable
)

SYSTEM_PROMPT = (
    "You are an RF engineering assistant in the US army. "
    "You have access to two tools:\n"
    "1. `check_current_rf_status`: Analyzes and provides the currently observed frequency hopping sequence and modulation scheme.\n"
    "2. `check_current_audio_status`: Checks audio signals to detect whether a drone is present. Use this whenever the user asks about audio, acoustics, or drone presence.\n\n"
    "Currently the US army employs the following two modulation and frequency hopping sequences. "
    "That is a legitate and legal RF signal would either have a frequency hopping sequence of [10,20] and would use BPSK modulation or it would have a frequency hopping sequence of [30,40] with QPSK modulation. "
    "If the modulation scheme is not QPSK or BPSK or if the frequency hopping sequence is not one of [10,20] or [30,40], you must warn the operator immediately. "
    "Based on the results from these tools, your task is to answer the questions so that even a civilian would understand what is going on."
)

# We use lazy initialization to create the async agent
global_agent = None

async def get_agent():
    global global_agent
    if global_agent is None:
        # Connect to our local SSE MCP server running on port 8000
        client = MultiServerMCPClient({
            "research": {
                "transport": "sse",
                "url": "http://localhost:8000/sse"
            }
        })
        mcp_tools = await client.get_tools()
        global_agent = create_react_agent(llm, mcp_tools)
    return global_agent

# --- 3. CHAT LOGIC ---
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Make sure you have your SYSTEM_PROMPT defined somewhere above this!
# SYSTEM_PROMPT = "You are an RF engineering assistant..."

async def chat_fn(message, history):
    if not message: return "Please enter a question."

    try:
        agent = await get_agent() # Or however you grab your global_agent

        # 1. Start the message list with your System Prompt!
        messages = [SystemMessage(content=SYSTEM_PROMPT)]

        # 2. Add the Gradio history (using the dictionary format)
        if history:
            for msg in history:
                if msg.get("role") == "user":
                    messages.append(HumanMessage(content=msg.get("content", "")))
                elif msg.get("role") == "assistant":
                    messages.append(AIMessage(content=msg.get("content", "")))

        # 3. Append the new user message (with hidden RF context if applicable)
        if sys_state.cur_analysis:
            rft_prompt = f" \nSTART_RFT Detected modulation is {sys_state.cur_analysis['mod'][0]}. Detected frequency hopping sequence is {str(sys_state.cur_analysis['freq_hop'])}. END_RFT"
            messages.append(HumanMessage(content=message + rft_prompt))
        else:
            messages.append(HumanMessage(content=message))

        # 4. Invoke the agent
        response = await agent.ainvoke({"messages": messages})

        return response["messages"][-1].content

    except Exception as e:
        return f"Error connecting to agent: {str(e)}"

async def respond(message, chat_history):
    if chat_history is None:
        chat_history = []

    bot_message = await chat_fn(message, chat_history)

    # 5. Append using the STRICT dictionary format required by your Gradio version
    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": bot_message})

    return "", chat_history

# --- INTERFACE ---
def stream(noise):
    sys_state.running = True
    while sys_state.running:
        generate_mixed_chunk(noise)
        fig, _ = update_plot(None, None)
        yield fig, "<h3>Monitoring...</h3>", gr.update(visible=True), gr.update(visible=True)
        plt.close(fig); time.sleep(2.0)

def pause(s,w):
    sys_state.running = False
    res = analyze_selection(s, w)
    sys_state.cur_analysis=res
    pause_fig,_=update_plot(s,w)
    return "<h3>Paused</h3>", gr.update(visible=True), gr.update(visible=True),pause_fig

def analyze(s, w):
  res = analyze_selection(s, w)
  sys_state.cur_analysis=res
  return update_plot(s, w)

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📡 3-Modulation Interceptor")
    gr.Markdown("Detect **BPSK (Blue)**, **QPSK (Green)**, and **16-QAM (Red/Threat)**.")
    with gr.Row():
        with gr.Column(scale=3):
          with gr.Row():
            plot = gr.Plot()
          with gr.Row():
            with gr.Column(scale=0.3):
              noise = gr.Slider(0, 0.5, 0.1, label="Noise")
              btn_play = gr.Button("▶ Start", variant="primary")
              btn_pause = gr.Button("⏸ Pause", variant="secondary")
              out = gr.HTML("Ready")
              s_start = gr.Slider(0, sys_state.chunk_duration-2*univ_hop_dur, 0.0, step=univ_hop_dur,label="Time", visible=True)
              s_width = gr.Slider(univ_hop_dur, 2*univ_hop_dur, univ_hop_dur,step=univ_hop_dur, label="Width", visible=True)
            with gr.Column(scale=1):
              plot2 = gr.Plot()
        with gr.Column(scale=1):
          chatbot = gr.Chatbot(label="ChatBot", height=500)
          msg = gr.Textbox(label="Your Message")
          clear = gr.Button("Clear Memory")

    btn_play.click(stream, [noise], [plot, out, s_start, s_width])
    btn_pause.click(pause, [s_start, s_width], [out, s_start, s_width, plot])
    s_start.change(analyze, [s_start, s_width], [plot, out])
    s_width.change(analyze, [s_start, s_width], [plot, out])

    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: None, None, chatbot, queue=False)

generate_mixed_chunk(0.1)
# if __name__ == "__main__": demo.queue().launch(share=True)
if __name__ == "__main__": demo.queue().launch(debug=True)

# LLM with composite signal

In [ ]:
!pip install -qU gradio nest-asyncio mcp langchain-openai langchain-mcp-adapters langgraph langchain-ollama

In [ ]:


def save_data(data_var,filename):
  with open(filename, 'wb') as file:
    # Use pickle.dump() to write the variable to the file
    pickle.dump(data_var, file)

def read_data(filename):
  with open(filename, 'rb') as f:
        # Load the object from the pickle file
        loaded_data = pickle.load(f)
  return loaded_data

class IQCNN(nn.Module):
    def __init__(self, num_classes=11):
        super(IQCNN, self).__init__()

        self.layer_dims=[]
        self.layers=nn.ModuleList()

        # Define convolutional layers here .......................................
        self.layers.append(nn.Conv1d(in_channels=2, out_channels=8, kernel_size=7, padding=3,bias=False))
        self.layers.append(nn.Conv1d(in_channels=8, out_channels=16, kernel_size=7, padding=3,bias=False))
        self.layers.append(nn.Conv1d(in_channels=16, out_channels=32, kernel_size=7, padding=3,bias=False))
        self.layers.append(nn.Conv1d(in_channels=32, out_channels=64, kernel_size=7, padding=3,bias=False))

        self.conv_num=len(self.layers)
        for i in range(self.conv_num):
          in_ch=self.layers[i].in_channels
          ker_sz=self.layers[i].kernel_size[0]
          self.layer_dims.append(in_ch*ker_sz)

        #Define linear layers here .....................................................
        self.layers.append(nn.Linear(64, 256,bias=False))
        self.layers.append(nn.Linear(256, num_classes,bias=False))

        for i in range(self.conv_num,len(self.layers)):
          self.layer_dims.append(self.layers[i].in_features)
        self.layer_dims.append(num_classes)
        # print(self.layer_dims)

        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)

        for i in range(self.conv_num):
          nn.init.xavier_uniform_(self.layers[i].weight)
        for i in range(self.conv_num,len(self.layers)-1):
            nn.init.kaiming_normal_(self.layers[i].weight, nonlinearity='relu')

        nn.init.kaiming_normal_(self.layers[i].weight, nonlinearity='linear')

    def forward(self, x):

        for i in range(self.conv_num):
         x=F.relu(self.layers[i](x))

        x = self.global_avg_pool(x)  # (batch_size, 64, 1)
        x = x.squeeze(-1)  # (batch_size, 64)

        for i in range(self.conv_num,len(self.layers)-1):
          x=F.relu(self.layers[i](x))
        x = self.layers[-1](x)  # (batch_size, output_shape)

        return x#F.log_softmax(x, dim=1)  # Use log_softmax for classification

def build_model(num_classes=3,model_name='IQCNN'):
  return IQCNN(num_classes=num_classes)#IQCNN(num_classes=11)

#.............................Client Class......................................

class Client:
  def __init__(self,dataset,num_classes=3,device='cpu'):
      self.device=device
      self.model = build_model(num_classes=num_classes).to(self.device)
      self.dataset =dataset
      self.dataloader = cycle(t.utils.data.DataLoader(self.dataset, batch_size=512, shuffle=True))
      self.cross_loss = nn.CrossEntropyLoss()
      self.optimizer = t.optim.Adam(self.model.parameters(), lr=0.001)

  def load_model(self,state_dict):
    self.model.load_state_dict(state_dict)

  def load_model_from_file(self,filename):
    state_dict=read_data(filename)
    self.load_model(state_dict)

  def save_model(self,filename):
    save_data(self.model.state_dict,filename)

  def client_loss(self,pred,y):
    return self.cross_loss(pred,y)

  def evaluate(self,test_loader):
    correct,total=0,0
    with t.no_grad():
        for data in test_loader:
            x, y = data
            x=x.to(self.device)
            y=y.to(self.device)
            output = self.model(x)
            for idx, i in enumerate(output):
                if t.argmax(i) == y[idx]:
                    correct +=1
                total +=1
    print(f'accuracy: {round(correct/total, 3)}')
    return round(correct/total, 3)


  def train_batch(self):
    x,y = next(self.dataloader)
    x=x.to(self.device)
    y=y.to(self.device)
    self.optimizer.zero_grad()
    pred = self.model(x)
    loss=self.client_loss(pred,y)
    loss.backward()
    self.optimizer.step()
    return loss.item()

  def train(self,epochs=int(1e3),echo=True,eval_acc=False):
    running_loss=0
    for epoch in range(epochs):
      epoch_loss = self.train_batch()
      running_loss+=epoch_loss

      if echo:
        if epoch%int(epochs/10) ==0 and epoch!=0:
          print("epoch:"+str(epoch)+" loss:"+ str(running_loss/int(epochs/10)))
          running_loss=0
          if eval_acc:
            self.evaluate(test_loader)
        # if epoch%(echo*10)==0 and epoch!=0:
        #   self.evaluate()

    return running_loss/int(epochs/10)

def create_synthetic_data(noise_level,num_datapoints=int(1e4)):
  fc = 40
  samples = 32
  duration = 0.1
  t_sym = np.linspace(0, duration, samples, endpoint=False)
  num_symbols=128

  # Scales for Unit Power
  scale_qpsk = 1.0 / np.sqrt(2)
  scale_16qam = 1.0 / np.sqrt(10)

  mod_options = ['BPSK', 'QPSK', '16-QAM']
  label_dict={'BPSK':0, 'QPSK':1, '16-QAM':2}
  data=[]
  labels=[]
  for _ in range(num_datapoints):
    m=np.random.choice(mod_options, 1)
    m=m[0]
    ground_truth = [m]*4 #128/4

    rx_i, rx_q = np.array([]), np.array([])
    for idx, mod in enumerate(ground_truth):
        # -- Normalized Transmitter --
        if mod == 'BPSK':
            s = (2*np.random.randint(0,2)-1) + 0j
        elif mod == 'QPSK':
            raw = (2*np.random.randint(0,2)-1) + 1j*(2*np.random.randint(0,2)-1)
            s = raw * scale_qpsk
        else:
            mp = np.array([-3, -1, 1, 3])
            raw = mp[np.random.randint(0,4)] + 1j*mp[np.random.randint(0,4)]
            s = raw * scale_16qam

        # Channel
        raw_wave = (s.real * np.cos(2*np.pi*fc*t_sym) - s.imag * np.sin(2*np.pi*fc*t_sym))

        # Add noise (Standard deviation)
        noise = noise_level * np.random.randn(len(raw_wave))
        noisy_chunk = raw_wave + noise

        # Rx - Match filter output
        rec_i = noisy_chunk * 2 * np.cos(2*np.pi*fc*t_sym)
        rec_q = noisy_chunk * -2 * np.sin(2*np.pi*fc*t_sym)
        rx_i=np.concatenate((rx_i,rec_i)); rx_q=np.concatenate((rx_q,rec_q))
    data.append(np.array([rx_i,rx_q]))
    labels.append(label_dict[m])

  X=np.array(data)
  Y=np.array(labels)
  print(X.shape,Y.shape)

  X_train_ar, X_test_ar, y_train_ar, y_test_ar = train_test_split(X, Y, test_size=0.3, random_state=42, stratify=Y) # split the data (70% train, 30% test)
  print(X_train_ar.shape, X_test_ar.shape, y_train_ar.shape, y_test_ar.shape)

  X_train = t.from_numpy(X_train_ar).float()
  y_train = t.from_numpy(y_train_ar).long() # Use long for integer labels
  X_test = t.from_numpy(X_test_ar).float()
  y_test = t.from_numpy(y_test_ar).long()
  trainset = TensorDataset(X_train, y_train)
  testset = TensorDataset(X_test, y_test)

  batch_size = 512

  train_loader = t.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)
  test_loader = t.utils.data.DataLoader(testset, batch_size=10, shuffle=True)

  return train_loader,test_loader,trainset,testset



In [ ]:
import itertools

def create_composite_data(noise_level,user_mods,num_datapoints=int(1e4)):
  fc = 40
  samples = 32
  duration = 0.1
  t_sym = np.linspace(0, duration, samples, endpoint=False)
  num_symbols=4


  # Scales for Unit Power
  scale_qpsk = 1.0 / np.sqrt(2)
  scale_16qam = 1.0 / np.sqrt(10)

  mod_options = ['BPSK', 'QPSK', '16-QAM']

  product_iterator=itertools.product(*user_mods)
  label_list=list(product_iterator)
  label_dict={'BPSK':0, 'QPSK':1, '16-QAM':2}
  label_dict={}
  for i,l in enumerate(label_list):
    label_dict[str(list(l))]=i
  num_classes=len(label_dict)
  data=[]
  labels=[]
  for dpt in range(num_datapoints):
    # print(dpt)
    composite_label=[]
    for u in range(len(user_mods)):
      m=np.random.choice(user_mods[u], 1)
      m=str(m[0])
      composite_label.append(m)

    ground_truth = [composite_label]*num_symbols #128/4
    rx_i, rx_q = np.array([]), np.array([])

    for idx, mods in enumerate(ground_truth):
        # -- Normalized Transmitter --
        s=0+0j
        for mod in mods:
          if mod == 'BPSK':
              s += (2*np.random.randint(0,2)-1) + 0j
          elif mod == 'QPSK':
              raw = (2*np.random.randint(0,2)-1) + 1j*(2*np.random.randint(0,2)-1)
              s += raw * scale_qpsk
          else:
              mp = np.array([-3, -1, 1, 3])
              raw = mp[np.random.randint(0,4)] + 1j*mp[np.random.randint(0,4)]
              s += raw * scale_16qam


        # Channel
        raw_wave = (s.real * np.cos(2*np.pi*fc*t_sym) - s.imag * np.sin(2*np.pi*fc*t_sym))

        # Add noise (Standard deviation)
        noise = noise_level * np.random.randn(len(raw_wave))
        noisy_chunk = raw_wave + noise

        # Rx - Match filter output
        rec_i = noisy_chunk * 2 * np.cos(2*np.pi*fc*t_sym)
        rec_q = noisy_chunk * -2 * np.sin(2*np.pi*fc*t_sym)

        rx_i=np.concatenate((rx_i,rec_i)); rx_q=np.concatenate((rx_q,rec_q))
        # rx_i=np.concatenate((rx_i,np.array([rec_i]))); rx_q=np.concatenate((rx_q,np.array([rec_q])))
    data.append(np.array([rx_i,rx_q]))
    labels.append(label_dict[str(composite_label)])

  X=np.array(data)
  Y=np.array(labels)
  print(X.shape,Y.shape)

  X_train_ar, X_test_ar, y_train_ar, y_test_ar = train_test_split(X, Y, test_size=0.3, random_state=42, stratify=Y) # split the data (70% train, 30% test)
  print(X_train_ar.shape, X_test_ar.shape, y_train_ar.shape, y_test_ar.shape)

  X_train = t.from_numpy(X_train_ar).float()
  y_train = t.from_numpy(y_train_ar).long() # Use long for integer labels
  X_test = t.from_numpy(X_test_ar).float()
  y_test = t.from_numpy(y_test_ar).long()
  trainset = TensorDataset(X_train, y_train)
  testset = TensorDataset(X_test, y_test)

  batch_size = 512

  train_loader = t.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)
  test_loader = t.utils.data.DataLoader(testset, batch_size=10, shuffle=True)

  return train_loader,test_loader,trainset,testset,num_classes,label_list

In [ ]:
user_mods=[['BPSK','QPSK'],
           ['QPSK','16-QAM']]
print(user_mods)
train_loader,test_loader,trainset,testset,num_composite_classes,label_list=create_composite_data(noise_level=0.1,user_mods=user_mods,num_datapoints=int(1e4))
device= 'cuda' if t.cuda.is_available() else 'cpu'
print(device)

In [ ]:
rf_client= Client(trainset,num_classes=num_composite_classes,device=device)
rf_client.train(epochs=int(1e4),echo=True,eval_acc=True)
rf_client.evaluate(test_loader)

In [ ]:
# 1. Install Gradio (Assuming this is run in a standard environment)
import torch
import gc
import os

import gradio as gr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import time
from collections import Counter

from PIL import Image

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
torch.cuda.empty_cache()

univ_hop_dur=0.1
univ_sph=8

# --- 1. CONFIGURATION ---
class SystemState:
    def __init__(self):
        self.fs = 320
        self.fc=40
        self.duration=0.1
        self.samples=32
        self.vec_sz=128
        self.current_rx_i = np.array([])
        self.current_rx_q = np.array([])

        self.chunk_duration = 4.0 #keep this a multiple of hop duration
        self.running = False
        self.current_waveform = np.array([])
        self.gt_iq=[]
        self.gt_mod=[]
        self.gt_fc=[]
        self.demod=[]
        self.cur_analysis=None

        # PROFILES (Now with BPSK!)
        self.profiles = {
            "S1 (Long Range)": {
                "freqs": [10, 20],
                "mod": "BPSK",
                "color": "#00CCFF", # Cyan/Blue
                "desc": "Robust / Low Rate"
            },
            "S2 (Standard)": {
                "freqs": [30, 40],
                "mod": "QPSK",
                "color": "#00FF00", # Green
                "desc": "Standard Link"
            },
            "S3 (THREAT)": {
                "freqs": [50, 60, 70],
                "mod": "16-QAM",
                "color": "#FF0000", # Red
                "desc": "High Speed Burst"
            }
        }

sys_state = SystemState()

# --- CLASSFIFER-----
def classify_normalized(i_val, q_val):
    point = i_val + 1j*q_val

    # --- Define NORMALIZED Constellations (Avg Power = 1.0) ---
    bpsk_pts = [-1, 1]

    scale_qpsk = 1.0 / np.sqrt(2)
    qpsk_pts = [scale_qpsk*(-1-1j), scale_qpsk*(-1+1j), scale_qpsk*(1-1j), scale_qpsk*(1+1j)]

    scale_16qam = 1.0 / np.sqrt(10)
    vals = [-3, -1, 1, 3]
    qam_pts = [scale_16qam*(x + 1j*y) for x in vals for y in vals]

    # --- Euclidean Distances ---
    err_bpsk = min([abs(point - p) for p in bpsk_pts])
    err_qpsk = min([abs(point - p) for p in qpsk_pts])
    err_16qam = min([abs(point - p) for p in qam_pts])

    # --- Penalties ---
    scores = {
        'BPSK': err_bpsk + 0.0,
        'QPSK': err_qpsk + 0.1,
        '16-QAM': err_16qam + 0.25
    }

    return min(scores, key=scores.get)

# --- 2. MIXED SIGNAL GENERATOR ---
def generate_mixed_chunk(noise_level):
    # fs = sys_state.fs
    samples = sys_state.samples
    duration=sys_state.duration
    fs=samples/duration
    total_dur = sys_state.chunk_duration

     # Scales for Unit Power
    scale_qpsk = 1.0 / np.sqrt(2)
    scale_16qam = 1.0 / np.sqrt(10)

    full_wave = []
    full_rx_i=[]
    full_rx_q=[]
    gt_mod=[]
    current_t = 0

    # Generate segments until buffer is full
    while current_t < total_dur:

        seg_dur=np.random.randint(0, 5)*duration+1.0
        if current_t + seg_dur > total_dur: seg_dur = total_dur - current_t



        # Generate Hopping
        num_symbols = int(np.ceil(seg_dur /duration))
        segment_wave = []


        fc = sys_state.fc
        t_sym = np.linspace(0, duration, samples, endpoint=False)

        composite_label=[]
        for u in range(len(user_mods)):
          m=np.random.choice(user_mods[u], 1)
          m=str(m[0])
          composite_label.append(m)

        ground_truth = [composite_label]*num_symbols #128/4
        rx_i, rx_q = np.array([]), np.array([])
        for idx, mods in enumerate(ground_truth):
            # -- Normalized Transmitter --
            s=0+0j
            for mod in mods:
              if mod == 'BPSK':
                  s += (2*np.random.randint(0,2)-1) + 0j
              elif mod == 'QPSK':
                  raw = (2*np.random.randint(0,2)-1) + 1j*(2*np.random.randint(0,2)-1)
                  s += raw * scale_qpsk
              else:
                  mp = np.array([-3, -1, 1, 3])
                  raw = mp[np.random.randint(0,4)] + 1j*mp[np.random.randint(0,4)]
                  s += raw * scale_16qam


            # Channel
            raw_wave = (s.real * np.cos(2*np.pi*fc*t_sym) - s.imag * np.sin(2*np.pi*fc*t_sym))

            # Add noise (Standard deviation)
            noise = noise_level * np.random.randn(len(raw_wave))
            noisy_chunk = raw_wave + noise

            # Rx - Match filter output
            rec_i = noisy_chunk * 2 * np.cos(2*np.pi*fc*t_sym)
            rec_q = noisy_chunk * -2 * np.sin(2*np.pi*fc*t_sym)
            rx_i=np.concatenate((rx_i,rec_i)); rx_q=np.concatenate((rx_q,rec_q))

            segment_wave.append(noisy_chunk)
            gt_mod.append(mods)

        if segment_wave:
          full_wave.append(np.concatenate(segment_wave))
          full_rx_i.append(rx_i)
          full_rx_q.append(rx_q)

        current_t += seg_dur

    raw = np.concatenate(full_wave)
    raw = raw[:int(fs*total_dur)]

    raw_i = np.concatenate(full_rx_i)
    raw_i = raw_i[:int(fs*total_dur)]

    raw_q = np.concatenate(full_rx_q)
    raw_q = raw_q[:int(fs*total_dur)]

    # sys_state.demod=np.concatenate(demod)

    # Add Noise
    sys_state.current_waveform = raw #+ (np.random.randn(len(raw)) * noise_level)
    sys_state.current_rx_i=raw_i
    sys_state.current_rx_q=raw_q
    # sys_state.gt_iq=gt_iq
    # sys_state.gt_fc=gt_fc
    sys_state.gt_mod=gt_mod

# --- 3. DECODER ---
def analyze_selection(start_t, width_t):
    fs = sys_state.fs
    wave = sys_state.current_waveform
    wave_i=sys_state.current_rx_i
    wave_q=sys_state.current_rx_q

    vec_sz=sys_state.vec_sz

    idx_start = int(start_t * fs)
    idx_end = int((start_t + width_t) * fs)
    idx_start = max(0, min(idx_start, len(wave)-1))
    idx_end = max(0, min(idx_end, len(wave)-1))

    if idx_end - idx_start < 50: return None

    chunk = wave[idx_start:idx_end]
    rx_i= wave_i[idx_start:idx_end]
    rx_q= wave_q[idx_start:idx_end]
    model_inp=np.stack([rx_i,rx_q],axis=0)
     #10 samples per symbol
    sps=sys_state.samples #int(univ_hop_dur*fs/univ_sph)
    gt_mod=sys_state.gt_mod[int(idx_start/sps):int(idx_end/sps)]
    gt_mod=list(map(str, gt_mod))

    num_chunks=int(width_t*fs/vec_sz)
    inp=np.array(np.split(model_inp,num_chunks,axis=1))
    inp=t.from_numpy(inp.astype(np.float32)).to(device)
    print(type(inp),inp.shape)
    pred=rf_client.model(inp)
    pred_list=list(t.argmax(pred,dim=1))
    pred_list=list(map(int, pred_list))
    pred_list=[label_list[pred_list[i]] for i in range(len(pred_list))]
    # pred_list=[]

    max_vote=Counter(pred_list).most_common(1)
    max_mod=max_vote[0]
    print(pred_list)
    print(max_mod)
    gt_max_mod=Counter(gt_mod).most_common(1)
    print('GT mod',gt_max_mod)

    return {
        "mod": max_mod,
        "gt_mod":gt_max_mod
    }

# --- 4. VISUALIZATION ---
def update_plot(scan_start=None, scan_width=None):
    if len(sys_state.current_waveform) == 0: generate_mixed_chunk(0.1)

    wave = sys_state.current_waveform
    fs = sys_state.fs

    fig = plt.figure(figsize=(10, 4))
    fig.patch.set_facecolor('#121212')
    plt.style.use('dark_background')

    ax_time = fig.add_subplot(111)

    # Standard Plots
    ax_time.plot(np.linspace(0, 4, len(wave)), wave, color='#00FF00', lw=0.5)
    ax_time.set_title("Time Domain", color='white')
    ax_time.set_ylim(-4, 4); ax_time.set_xlim(0, 4)

    status_html = "<h3>Scanning...</h3>"

    if scan_start is not None:
        # Highlight Box
        rect_s = patches.Rectangle((scan_start, -100), scan_width, 300, edgecolor='white', facecolor='yellow', alpha=0.5)
        ax_time.add_patch(rect_s)

    return fig, status_html

# ---- CHATBOT -----

import nest_asyncio
import threading
import asyncio
from mcp.server.fastmcp import FastMCP
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage

# Apply nest_asyncio to allow nested event loops in Colab/Jupyter/Gradio
nest_asyncio.apply()

# --- 1. SET UP FAST MCP SERVER ---
mcp = FastMCP("RF_Analysis_Assistant")

@mcp.tool()
def check_current_rf_status() -> str:
    """Reads the current RF environment variables including modulation and frequency hopping."""
    if sys_state.cur_analysis is not None:
        mod = sys_state.cur_analysis['mod'][0]
        # freq_hop = str(sys_state.cur_analysis['freq_hop'])
        return f"START_RFT Detected modulation schemes are {mod}. END_RFT"
    return "No RF analysis available. The operator has not scanned a signal yet."

@mcp.tool()
def check_current_audio_status() -> str:
  return "no drone detected in the audio signal"
  # """Checks audio signals to detect whether a drone is present."""
  # from torch.utils.data import RandomSampler
  # time_start = 1
  # time_end = 20
  # result = []
  # num_samples = int(time_end-time_start)+1
  # random_sampler = RandomSampler(test_dataset, num_samples=num_samples, replacement=False)
  # sampled_dataloader = DataLoader(test_dataset, batch_size=1, sampler=random_sampler)
  # model.eval()
  # with t.no_grad():
  #   for embeddings, labels in sampled_dataloader:
  #     embeddings, labels = embeddings.to(device), labels.to(device)
  #     output = model(embeddings)
  #     result.append((output > 0.5).float())
  # print(result)
  # if any(pred.item() == 1.0 for pred in result):
  #   return 'The current audio signal was checked and a drone was detected!'
  # else:
  #   return 'The current audio signal was checked and no drone was detected!'

def start_server():
    print("Starting MCP Server on SSE transport...")
    mcp.run(transport="sse")

# Start MCP Server in the background
server_thread = threading.Thread(target=start_server, name="mcp_server_thread", daemon=True)
server_thread.start()

!sudo apt-get install -y zstd

# 1. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Start Ollama server in the background
import threading
import subprocess
import time

def run_ollama():
    # 'ollama serve' starts the API server
    subprocess.run(["ollama", "serve"])

# Launch the server thread
ollama_thread = threading.Thread(target=run_ollama, daemon=True)
ollama_thread.start()

# Wait a few seconds for the server to wake up
time.sleep(5)
print("✅ Ollama server is running!")
!ollama pull llama3.2

# --- 2. SET UP LANGGRAPH & OLLAMA ---
# Initialize the LLM (Ensure 'ollama serve' is running in your environment!)
from langchain_ollama import ChatOllama

# ChatOllama is specifically built to handle Llama 3.2's tool-calling quirks
llm = ChatOllama(
    model="llama3.2",
    temperature=0 # Keep temperature at 0 to make tool calling more deterministic and stable
)

SYSTEM_PROMPT = (
    "You are an RF engineering assistant in the US army. "
    "You have access to two tools:\n"
    "1. `check_current_rf_status`: Analyzes and provides the currently observed  modulation schemes.\n"
    "2. `check_current_audio_status`: Checks audio signals to detect whether a drone is present. Use this whenever the user asks about audio, acoustics, or drone presence.\n\n"
    "Currently the US army employs the following two modulation schemes: QPSK or BPSK. "
    "That is a legitate and legal RF signal would either have a QPSK modulation or a BPSK modulation "
    "If the observed modulation schemes contain something other than QPSK or BPSK you must warn the user immediately. "
    "Based on the results from these tools, your task is to answer the questions so that even a civilian would understand what is going on."
)

# We use lazy initialization to create the async agent
global_agent = None

async def get_agent():
    global global_agent
    if global_agent is None:
        # Connect to our local SSE MCP server running on port 8000
        client = MultiServerMCPClient({
            "research": {
                "transport": "sse",
                "url": "http://localhost:8000/sse"
            }
        })
        mcp_tools = await client.get_tools()
        global_agent = create_react_agent(llm, mcp_tools)
    return global_agent

# --- 3. CHAT LOGIC ---
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Make sure you have your SYSTEM_PROMPT defined somewhere above this!
# SYSTEM_PROMPT = "You are an RF engineering assistant..."

async def chat_fn(message, history):
    if not message: return "Please enter a question."

    try:
        agent = await get_agent() # Or however you grab your global_agent

        # 1. Start the message list with your System Prompt!
        messages = [SystemMessage(content=SYSTEM_PROMPT)]

        # 2. Add the Gradio history (using the dictionary format)
        if history:
            for msg in history:
                if msg.get("role") == "user":
                    messages.append(HumanMessage(content=msg.get("content", "")))
                elif msg.get("role") == "assistant":
                    messages.append(AIMessage(content=msg.get("content", "")))

        # 3. Append the new user message (with hidden RF context if applicable)
        if sys_state.cur_analysis:
            rft_prompt = f" \nSTART_RFT Detected modulation is {sys_state.cur_analysis['mod'][0]} END_RFT"
            messages.append(HumanMessage(content=message + rft_prompt))
        else:
            messages.append(HumanMessage(content=message))

        # 4. Invoke the agent
        response = await agent.ainvoke({"messages": messages})

        return response["messages"][-1].content

    except Exception as e:
        return f"Error connecting to agent: {str(e)}"

async def respond(message, chat_history):
    if chat_history is None:
        chat_history = []

    bot_message = await chat_fn(message, chat_history)

    # 5. Append using the STRICT dictionary format required by your Gradio version
    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": bot_message})

    return "", chat_history

# --- INTERFACE ---
def stream(noise):
    sys_state.running = True
    while sys_state.running:
        generate_mixed_chunk(noise)
        fig, _ = update_plot(None, None)
        yield fig, "<h3>Monitoring...</h3>", gr.update(visible=True), gr.update(visible=True)
        plt.close(fig); time.sleep(2.0)

def pause(s,w):
    sys_state.running = False
    res = analyze_selection(s, w)
    sys_state.cur_analysis=res
    pause_fig,_=update_plot(s,w)
    return "<h3>Paused</h3>", gr.update(visible=True), gr.update(visible=True),pause_fig

def analyze(s, w):
  res = analyze_selection(s, w)
  sys_state.cur_analysis=res
  return update_plot(s, w)

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📡 3-Modulation Interceptor")
    gr.Markdown("Detect **BPSK (Blue)**, **QPSK (Green)**, and **16-QAM (Red/Threat)**.")
    with gr.Row():
        with gr.Column(scale=3):
          with gr.Row():
            plot = gr.Plot()
          with gr.Row():
            with gr.Column(scale=0.3):
              noise = gr.Slider(0, 0.5, 0.1, label="Noise")
              btn_play = gr.Button("▶ Start", variant="primary")
              btn_pause = gr.Button("⏸ Pause", variant="secondary")
              out = gr.HTML("Ready")
              s_start = gr.Slider(0, sys_state.chunk_duration-0.8, 0.0, step=0.4,label="Time", visible=True)
              s_width = gr.Slider(0.4, 0.8, 0.4,step=0.4, label="Width", visible=True)
            with gr.Column(scale=1):
              plot2 = gr.Plot()
        with gr.Column(scale=1):
          chatbot = gr.Chatbot(label="ChatBot", height=500)
          msg = gr.Textbox(label="Your Message")
          clear = gr.Button("Clear Memory")

    btn_play.click(stream, [noise], [plot, out, s_start, s_width])
    btn_pause.click(pause, [s_start, s_width], [out, s_start, s_width, plot])
    s_start.change(analyze, [s_start, s_width], [plot, out])
    s_width.change(analyze, [s_start, s_width], [plot, out])

    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: None, None, chatbot, queue=False)

generate_mixed_chunk(0.1)
# if __name__ == "__main__": demo.queue().launch(share=True)
if __name__ == "__main__": demo.queue().launch(debug=True)

# New Interface


In [ ]:
!git clone https://github.com/saraalemadi/DroneAudioDataset.git

In [ ]:
%cd DroneAudioDataset

In [ ]:
%%writefile train_cnn.py
import os
import math
import random
import argparse
from dataclasses import dataclass
from typing import List, Tuple, Dict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm



# ----------------------------
# Repro
# ----------------------------
def seed_everything(seed: int = 1337) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ----------------------------
# Dataset indexing
# ----------------------------
AUDIO_EXTS = (".wav", ".flac", ".mp3", ".ogg", ".m4a")


def list_class_folders(data_dir: str) -> List[str]:
    classes = []
    for name in sorted(os.listdir(data_dir)):
        p = os.path.join(data_dir, name)
        if os.path.isdir(p) and not name.startswith("."):
            classes.append(name)
    if not classes:
        raise RuntimeError(f"No class folders found in: {data_dir}")
    return classes


def index_files(data_dir: str) -> Tuple[List[str], List[int], List[str], Dict[str, int]]:
    classes = list_class_folders(data_dir)
    class_to_idx = {c: i for i, c in enumerate(classes)}

    paths: List[str] = []
    labels: List[int] = []

    for c in classes:
        cdir = os.path.join(data_dir, c)
        for root, _, files in os.walk(cdir):
            for fn in files:
                if fn.lower().endswith(AUDIO_EXTS):
                    paths.append(os.path.join(root, fn))
                    labels.append(class_to_idx[c])

    if not paths:
        raise RuntimeError(f"No audio files found under: {data_dir}")

    return paths, labels, classes, class_to_idx
# ----------------------------
# Audio -> log-mel + aug
# ----------------------------
@dataclass
class AudioConfig:
    sample_rate: int = 16000
    clip_seconds: float = 2.5  # fixed length clips
    n_fft: int = 1024
    hop_length: int = 256
    n_mels: int = 64
    f_min: int = 30
    f_max: int = 8000
    time_mask_param: int = 24
    freq_mask_param: int = 10


class DroneAudioDataset(Dataset):
    def __init__(
        self,
        paths: List[str],
        labels: List[int],
        cfg: AudioConfig,
        train: bool = True,
    ):
        self.paths = paths
        self.labels = labels
        self.cfg = cfg
        self.train = train

        self.melspec = torchaudio.transforms.MelSpectrogram(
            sample_rate=cfg.sample_rate,
            n_fft=cfg.n_fft,
            hop_length=cfg.hop_length,
            n_mels=cfg.n_mels,
            f_min=cfg.f_min,
            f_max=cfg.f_max,
            power=2.0,
        )
        self.to_db = torchaudio.transforms.AmplitudeToDB(stype="power")

        # SpecAugment-style
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=cfg.time_mask_param)
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=cfg.freq_mask_param)

    def _load_resample_mono(self, path: str) -> torch.Tensor:
        wav, sr = torchaudio.load(path)  # [C, T]
        if wav.size(0) > 1:
            wav = wav.mean(dim=0, keepdim=True)  # mono
        if sr != self.cfg.sample_rate:
            wav = torchaudio.transforms.Resample(sr, self.cfg.sample_rate)(wav)
        return wav  # [1, T]

    def _crop_or_pad(self, wav: torch.Tensor) -> torch.Tensor:
        target_len = int(self.cfg.sample_rate * self.cfg.clip_seconds)
        T = wav.size(1)
        if T == target_len:
            return wav
        if T > target_len:
            # random crop during train, center crop during eval
            if self.train:
                start = random.randint(0, T - target_len)
            else:
                start = (T - target_len) // 2
            return wav[:, start : start + target_len]
        # pad
        pad = target_len - T
        return F.pad(wav, (0, pad), mode="constant", value=0.0)

    def _wav_to_logmel(self, wav: torch.Tensor) -> torch.Tensor:
        # wav: [1, T]
        mel = self.melspec(wav)         # [1, n_mels, frames]
        logmel = self.to_db(mel)        # dB
        # normalize per-sample
        logmel = (logmel - logmel.mean()) / (logmel.std() + 1e-6)
        return logmel                   # [1, n_mels, frames]

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, i: int):
        path = self.paths[i]
        y = self.labels[i]

        wav = self._load_resample_mono(path)
        wav = self._crop_or_pad(wav)

        # light waveform augmentation
        if self.train:
            # random gain
            gain = 10 ** random.uniform(-0.15, 0.15)
            wav = wav * gain
            # tiny gaussian noise
            if random.random() < 0.25:
                wav = wav + 0.003 * torch.randn_like(wav)

        x = self._wav_to_logmel(wav)  # [1, M, F]

        if self.train:
            if random.random() < 0.7:
                x = self.freq_mask(x)
            if random.random() < 0.7:
                x = self.time_mask(x)

        return x, y
# ----------------------------
# CNN model
# ----------------------------
class SmallAudioCNN(nn.Module):
    def __init__(self, n_classes: int):
        super().__init__()
        # input: [B, 1, n_mels, frames]
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.25),
            nn.Linear(128, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.classifier(x)
        return x
# ----------------------------
# Train / eval
# ----------------------------
@torch.no_grad()
def evaluate(model, loader, device) -> Tuple[float, np.ndarray, np.ndarray]:
    model.eval()
    loss_fn = nn.CrossEntropyLoss()

    total_loss = 0.0
    n = 0
    all_y = []
    all_p = []

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)
        loss = loss_fn(logits, y)

        total_loss += loss.item() * x.size(0)
        n += x.size(0)

        preds = torch.argmax(logits, dim=1)
        all_y.append(y.cpu().numpy())
        all_p.append(preds.cpu().numpy())

    return total_loss / max(n, 1), np.concatenate(all_y), np.concatenate(all_p)
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", type=str, default=".", help="Path to Multiclass_Drone_Audio (contains class folders)")
    ap.add_argument("--epochs", type=int, default=25)
    ap.add_argument("--batch_size", type=int, default=32)
    ap.add_argument("--lr", type=float, default=2e-3)
    ap.add_argument("--seed", type=int, default=1337)
    ap.add_argument("--num_workers", type=int, default=2)
    ap.add_argument("--val_frac", type=float, default=0.15)
    ap.add_argument("--test_frac", type=float, default=0.15)
    args = ap.parse_args()

    seed_everything(args.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    paths, labels, classes, _ = index_files(args.data_dir)
    labels_np = np.array(labels)

    # split train/val/test stratified
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=args.test_frac, random_state=args.seed)
    trainval_idx, test_idx = next(sss1.split(np.zeros_like(labels_np), labels_np))

    labels_trainval = labels_np[trainval_idx]
    val_size = args.val_frac / (1.0 - args.test_frac)

    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_size, random_state=args.seed)
    train_idx, val_idx_rel = next(sss2.split(np.zeros_like(labels_trainval), labels_trainval))
    val_idx = trainval_idx[val_idx_rel]
    train_idx = trainval_idx[train_idx]

    def subset(idxs):
        return [paths[i] for i in idxs], [labels[i] for i in idxs]

    train_paths, train_labels = subset(train_idx)
    val_paths, val_labels = subset(val_idx)
    test_paths, test_labels = subset(test_idx)

    cfg = AudioConfig()
    train_ds = DroneAudioDataset(train_paths, train_labels, cfg, train=True)
    val_ds = DroneAudioDataset(val_paths, val_labels, cfg, train=False)
    test_ds = DroneAudioDataset(test_paths, test_labels, cfg, train=False)

    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True,
                              num_workers=args.num_workers, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False,
                            num_workers=args.num_workers, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=args.batch_size, shuffle=False,
                             num_workers=args.num_workers, pin_memory=True)

    model = SmallAudioCNN(n_classes=len(classes)).to(device)

    # handle imbalance (optional but usually helps)
    counts = np.bincount(np.array(train_labels), minlength=len(classes))
    weights = (counts.sum() / (counts + 1e-9))
    weights = weights / weights.mean()
    class_weights = torch.tensor(weights, dtype=torch.float32, device=device)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)

    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs)

    best_val = float("inf")

    best_path = "/content/drive/MyDrive/ARL/best_cnn.pt"
    ckpt = torch.load(best_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    test_loss, ty, tp = evaluate(model, test_loader, device)
    test_acc = (tp == ty).mean()

    print("\n=== TEST ===")
    print(f"test_loss={test_loss:.4f}  test_acc={test_acc:.3f}")
    print("\nClasses:", classes)
    print("\nClassification report:\n", classification_report(ty, tp, target_names=classes, digits=3))

    #archiving training in case we need it later
    '''
    for epoch in range(1, args.epochs + 1):
        model.train()
        running = 0.0
        n = 0

        pbar = tqdm(train_loader, desc=f"epoch {epoch}/{args.epochs}", leave=False)
        for x, y in pbar:
            x = x.to(device)
            y = y.to(device)

            opt.zero_grad(set_to_none=True)
            logits = model(x)
            loss = loss_fn(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()

            running += loss.item() * x.size(0)
            n += x.size(0)
            pbar.set_postfix(loss=running / max(n, 1))

        sched.step()

        val_loss, vy, vp = evaluate(model, val_loader, device)
        val_acc = (vp == vy).mean()

        print(f"[epoch {epoch}] train_loss={running/max(n,1):.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.3f}")

        if val_loss < best_val:
            best_val = val_loss
            torch.save(
                {"model": model.state_dict(), "classes": classes, "cfg": cfg.__dict__},
                best_path,
            )
            print(f"  saved: {best_path}")

    # Test with best checkpoint
    ckpt = torch.load(best_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    test_loss, ty, tp = evaluate(model, test_loader, device)
    test_acc = (tp == ty).mean()

    print("\n=== TEST ===")
    print(f"test_loss={test_loss:.4f}  test_acc={test_acc:.3f}")
    print("\nClasses:", classes)
    print("\nClassification report:\n", classification_report(ty, tp, target_names=classes, digits=3))
    '''
if __name__ == "__main__":
    main()

In [ ]:
!python train_cnn.py --data_dir Multiclass_Drone_Audio --epochs 10 --batch_size 32

In [ ]:
import torch
import torchaudio
import torch.nn as nn
import random
import os
from IPython.display import Audio

device = "cuda" if torch.cuda.is_available() else "cpu"

# CNN architecture (same as training)
class SmallAudioCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16,32,3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten(),
            nn.Dropout(0.25), # Added the missing Dropout layer
            nn.Linear(128,n_classes)
        )

    def forward(self,x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
!pip install -qU gradio nest-asyncio mcp langchain-openai langchain-mcp-adapters langgraph langchain-ollama

In [ ]:
# !sudo apt-get install -y zstd

# # 1. Install Ollama
# !curl -fsSL https://ollama.com/install.sh | sh

# # 2. Start Ollama server in the background
# import threading
# import subprocess
# import time

# def run_ollama():
#     # 'ollama serve' starts the API server
#     subprocess.run(["ollama", "serve"])

# # Launch the server thread
# ollama_thread = threading.Thread(target=run_ollama, daemon=True)
# ollama_thread.start()

# # Wait a few seconds for the server to wake up
# time.sleep(5)
# print("✅ Ollama server is running!")
# !ollama pull llama3.2

In [ ]:
#@title { vertical-output: true}

import torch
import gc
import os
import io
import time
import threading
import subprocess
import asyncio
from collections import Counter

import gradio as gr
import numpy as np
import matplotlib.pyplot as plt
import nest_asyncio
from io import BytesIO
from PIL import Image
from mcp.server.fastmcp import FastMCP
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_ollama import ChatOllama

import base64
import random

from google.colab import drive
drive.mount('/content/drive')


os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
torch.cuda.empty_cache()

univ_hop_dur = 0.8
univ_sph = 8


IMAGE_PATH = "/content/drive/MyDrive/ARL/dca.png"
DRONE_IMAGE_PATH = "/content/drive/MyDrive/ARL/drone.png"
IS_DRONE_IN_RANGE = False
IS_DRONE_AUDIO_RANGE = False
IMAGE_WIDTH_PX = 500
IMAGE_HEIGHT_PX = 693

SENSOR_RADIUS_PX = 50
NUM_SENSORS = 25
DRONE_RADIUS_PX = 10  # since the drone is 20x20

#convert white background in drone to alpha
from PIL import Image
import os
import base64
from io import BytesIO

def load_drone_as_data_uri(path, white_thresh=200):
    if not os.path.exists(path):
        return ""

    img = Image.open(path).convert("RGBA")
    data = img.getdata()

    new_data = []
    for r, g, b, a in data:
        if r >= white_thresh and g >= white_thresh and b >= white_thresh:
            new_data.append((255, 255, 255, 0))
        else:
            new_data.append((r, g, b, a))

    img.putdata(new_data)

    buffer = BytesIO()
    img.save(buffer, format="PNG")
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{encoded}"

#Drone Movement Parameters
CURRENT_POS = (
    random.randint(DRONE_RADIUS_PX, IMAGE_WIDTH_PX - DRONE_RADIUS_PX),
    random.randint(DRONE_RADIUS_PX, IMAGE_HEIGHT_PX - DRONE_RADIUS_PX),
)
LAST_MOVE_TIME = None

SENSORS = [
    (235, 37),
    (190, 89),
    (167, 135),
    (190, 221),
    (165, 287),
    (179, 350),
    (164, 417),
    (106, 462),
    (105, 550),
    (161, 593),
    (226, 633),
    (307, 650),
    (383, 617),
    (403, 551),
    (420, 480),
    (431, 421),
    (435, 345),
    (429, 285),
    (412, 224),
    (394, 177),
    (373, 128),
    (332, 86),
    (305, 38)
]

# --- 1. CONFIGURATION ---
class SystemState:
    def __init__(self):
        self.fs = 4000
        self.chunk_duration = 4.0
        self.running = False
        self.current_waveform = np.array([])
        self.gt_iq = []
        self.gt_mod = []
        self.gt_fc = []
        self.demod = []
        self.cur_analysis = None
        self.cur_spec_base64 = None

        self.profiles = {
            "S1 (Long Range)": {
                "freqs": [10, 20],
                "mod": "BPSK",
                "color": "#00CCFF",
                "desc": "Robust / Low Rate"
            },
            "S2 (Standard)": {
                "freqs": [30, 40],
                "mod": "QPSK",
                "color": "#00FF00",
                "desc": "Standard Link"
            },
            "S3 (THREAT)": {
                "freqs": [50, 60, 70],
                "mod": "16-QAM",
                "color": "#FF0000",
                "desc": "High Speed Burst"
            }
        }



sys_state = SystemState()

# --- CLASSIFIER ---
def classify_normalized(i_val, q_val):
    point = i_val + 1j * q_val

    bpsk_pts = [-1, 1]

    scale_qpsk = 1.0 / np.sqrt(2)
    qpsk_pts = [
        scale_qpsk * (-1 - 1j),
        scale_qpsk * (-1 + 1j),
        scale_qpsk * (1 - 1j),
        scale_qpsk * (1 + 1j),
    ]

    scale_16qam = 1.0 / np.sqrt(10)
    vals = [-3, -1, 1, 3]
    qam_pts = [scale_16qam * (x + 1j * y) for x in vals for y in vals]

    err_bpsk = min(abs(point - p) for p in bpsk_pts)
    err_qpsk = min(abs(point - p) for p in qpsk_pts)
    err_16qam = min(abs(point - p) for p in qam_pts)

    scores = {
        "BPSK": err_bpsk + 0.0,
        "QPSK": err_qpsk + 0.1,
        "16-QAM": err_16qam + 0.25,
    }

    return min(scores, key=scores.get)


# --- 2. MIXED SIGNAL GENERATOR ---
def generate_mixed_chunk(noise_level):
    fs = sys_state.fs
    total_dur = sys_state.chunk_duration

    full_wave = []
    gt_iq = []
    gt_mod = []
    gt_fc = []
    demod = []
    current_t = 0

    while current_t < total_dur:
        hop_dur = univ_hop_dur
        seg_dur = np.random.randint(0, 5) * hop_dur + 1.0
        if current_t + seg_dur > total_dur:
            seg_dur = total_dur - current_t

        p_name = np.random.choice(list(sys_state.profiles.keys()))
        profile = sys_state.profiles[p_name]

        freq_list = profile["freqs"]
        mod_type = profile["mod"]

        num_hops = int(np.ceil(seg_dur / hop_dur))
        segment_wave = []

        for i in range(num_hops):
            this_hop_dur = hop_dur
            if this_hop_dur <= 0:
                break

            f_c = freq_list[i % len(freq_list)]
            t = np.linspace(0, this_hop_dur, int(fs * this_hop_dur), endpoint=False)

            num_syms = univ_sph

            if mod_type == "16-QAM":
                scale = 1.0 / np.sqrt(10)
                levs = [-3, -1, 1, 3]
                syms = [
                    complex(x, y) * scale
                    for x, y in zip(
                        np.random.choice(levs, num_syms),
                        np.random.choice(levs, num_syms),
                    )
                ]
            elif mod_type == "QPSK":
                scale = 1.0 / np.sqrt(2)
                syms = [
                    (np.random.choice([-1, 1]) + 1j * np.random.choice([-1, 1])) * scale
                    for _ in range(num_syms)
                ]
            else:
                syms = [(2 * np.random.randint(0, 2) - 1) + 0j for _ in range(num_syms)]

            gt_iq += syms
            gt_fc += [f_c] * len(syms)
            gt_mod += [mod_type] * len(syms)

            s_up = np.repeat(syms, int(len(t) / num_syms) + 1)[: len(t)]
            wave = s_up.real * np.cos(2 * np.pi * f_c * t) - s_up.imag * np.sin(2 * np.pi * f_c * t)

            demod_i = wave * 2 * np.cos(2 * np.pi * f_c * t)
            demod_q = -wave * 2 * np.sin(2 * np.pi * f_c * t)

            demod_i = np.array(np.array_split(demod_i, num_syms))
            demod_q = np.array(np.array_split(demod_q, num_syms))

            demod_i = np.mean(demod_i, axis=1)
            demod_q = np.mean(demod_q, axis=1)

            demod.append(demod_i + 1j * demod_q)
            segment_wave.append(wave)

        if segment_wave:
            full_wave.append(np.concatenate(segment_wave))

        current_t += seg_dur

    raw = np.concatenate(full_wave)
    raw = raw[: int(fs * total_dur)]

    sys_state.demod = np.concatenate(demod)
    sys_state.current_waveform = raw + (np.random.randn(len(raw)) * noise_level)
    sys_state.gt_iq = gt_iq
    sys_state.gt_fc = gt_fc
    sys_state.gt_mod = gt_mod


# --- 3. DECODER ---
def analyze_selection(start_t, width_t):
    fs = sys_state.fs
    wave = sys_state.current_waveform

    idx_start = int(start_t * fs)
    idx_end = int((start_t + width_t) * fs)
    idx_start = max(0, min(idx_start, len(wave) - 1))
    idx_end = max(0, min(idx_end, len(wave) - 1))

    if idx_end - idx_start < 50:
        return None

    chunk = wave[idx_start:idx_end]
    sps = int(univ_hop_dur * fs / univ_sph)

    gt_iq = sys_state.gt_iq[int(idx_start / sps):int(idx_end / sps)]
    gt_fc = sys_state.gt_fc[int(idx_start / sps):int(idx_end / sps)]
    gt_mod = sys_state.gt_mod[int(idx_start / sps):int(idx_end / sps)]
    demod = sys_state.demod[int(idx_start / sps):int(idx_end / sps)]

    fft_vals = np.abs(np.fft.fft(chunk))
    fft_freqs = np.fft.fftfreq(len(chunk), 1 / fs)
    pos = fft_freqs > 0

    peak_idx = np.argmax(fft_vals[pos])
    dom_freq = fft_freqs[pos][peak_idx]

    threshold = np.max(fft_vals[pos]) * 0.4
    peaks = fft_freqs[pos][fft_vals[pos] > threshold]
    detected_freqs = sorted(list(set([int(round(f / 10) * 10) for f in peaks if f > 10])))

    best_match = "Unknown"
    best_score = 0
    match_color = "gray"
    best_freq_list = []

    for name, profile in sys_state.profiles.items():
        common = len(set(profile["freqs"]).intersection(set(detected_freqs)))
        if common > best_score:
            best_score = common
            best_match = name
            match_color = profile["color"]
            best_freq_list = profile["freqs"]

    samples_per_sym = int(fs * univ_hop_dur / univ_sph)
    num_syms_in_chunk = int(len(chunk) / samples_per_sym)
    sym_chunks = np.array_split(chunk, num_syms_in_chunk)
    t_sym = np.linspace(0, univ_hop_dur / univ_sph, samples_per_sym, endpoint=False)

    pred_list = []
    freq_hop_seq = []
    rec_constellation = []

    for sym_chunk in sym_chunks:
        best_val = -np.inf
        best_freq = None
        best_i = 0
        best_q = 0

        for fc in best_freq_list:
            rec_i = np.mean(sym_chunk * 2 * np.cos(2 * np.pi * fc * t_sym))
            rec_q = np.mean(sym_chunk * -2 * np.sin(2 * np.pi * fc * t_sym))
            val = rec_i**2 + rec_q**2
            if val > best_val:
                best_val = val
                best_freq = fc
                best_i = rec_i
                best_q = rec_q

        freq_hop_seq.append(best_freq)
        pred_mod = classify_normalized(best_i, best_q)
        pred_list.append(pred_mod)
        rec_constellation.append(best_i + 1j * best_q)

    max_vote = Counter(pred_list).most_common(1)
    max_mod = max_vote[0]
    constellation = np.array(rec_constellation)

    return {
        "seq": best_match,
        "color": match_color,
        "iq": constellation,
        "freqs": detected_freqs,
        "dom_freq": dom_freq,
        "mod": max_mod,
        "freq_hop": best_freq_list,
    }


# --- 4. STATUS CIRCLE + VIS PANEL ---

#helper functions
def point_to_segment_distance(px, py, x1, y1, x2, y2):
        dx = x2 - x1
        dy = y2 - y1

        if dx == 0 and dy == 0:
            return ((px - x1) ** 2 + (py - y1) ** 2) ** 0.5

        t = ((px - x1) * dx + (py - y1) * dy) / (dx * dx + dy * dy)
        t = max(0.0, min(1.0, t))

        closest_x = x1 + t * dx
        closest_y = y1 + t * dy

        return ((px - closest_x) ** 2 + (py - closest_y) ** 2) ** 0.5

# --- RANDOM PATH GENERATION IN PIXELS ---
def rand_pos():
    step = 40  # max movement per refresh, in pixels
    new_x = CURRENT_POS[0] + random.randint(-step, step)
    new_y = CURRENT_POS[1] + random.randint(-step, step)

    new_x = max(DRONE_RADIUS_PX, min(new_x, IMAGE_WIDTH_PX - DRONE_RADIUS_PX))
    new_y = max(DRONE_RADIUS_PX, min(new_y, IMAGE_HEIGHT_PX - DRONE_RADIUS_PX))
    return (new_x, new_y)

def current_threat_active():
    if not sys_state.gt_mod:
        return False
    return "16-QAM" in sys_state.gt_mod


def drone_movement_and_detection():
    global CURRENT_POS, LAST_MOVE_TIME
    global IS_DRONE_IN_RANGE, IS_DRONE_AUDIO_RANGE

    now = time.time()

    if LAST_MOVE_TIME is None:
        dt = 2.0
    else:
        dt = now - LAST_MOVE_TIME

    LAST_MOVE_TIME = now

    p1 = CURRENT_POS
    p2 = rand_pos()
    CURRENT_POS = p2

    is_threat = current_threat_active()

    sensor_states = []
    any_triggered = False
    audio_triggered = False

    for sx, sy in SENSORS:
        dist = point_to_segment_distance(
            sx, sy,
            p1[0], p1[1],
            p2[0], p2[1]
        )

        triggered = is_threat and (dist <= SENSOR_RADIUS_PX)
        a_triggered = (dist <= SENSOR_RADIUS_PX)

        any_triggered = any_triggered or triggered
        audio_triggered = audio_triggered or a_triggered

        sensor_states.append({
            "x": sx,
            "y": sy,
            "triggered": triggered,
            "audio_triggered": a_triggered,
        })

    IS_DRONE_IN_RANGE = any_triggered
    IS_DRONE_AUDIO_RANGE = audio_triggered

    return {
        "p1": p1,
        "p2": p2,
        "dt": dt,
        "is_threat": is_threat,
        "sensor_states": sensor_states,
    }

def build_status_circle_html():
    movement = drone_movement_and_detection()
    movement_duration = max(0.05, movement["dt"])

    p1 = movement["p1"]
    p2 = movement["p2"]
    is_threat = movement["is_threat"]
    sensor_states = movement["sensor_states"]

    drone_color = "#ff2b2bff" if is_threat else "#22c55e00"
    animation_name = f"moveDot{random.randint(0, 100000)}"

    # --- LOAD IMAGES ---
    #background
    img_src = ""
    if os.path.exists(IMAGE_PATH):
        with open(IMAGE_PATH, "rb") as f:
            encoded = base64.b64encode(f.read()).decode("utf-8")
        ext = IMAGE_PATH.split(".")[-1].lower()
        if ext == "jpg":
            ext = "jpeg"
        img_src = f"data:image/{ext};base64,{encoded}"
    #drone
    drone_img_src = load_drone_as_data_uri(DRONE_IMAGE_PATH)

    #drone rendering
    drone_filter = (
      "brightness(0) invert(0.6)"  # grey
      if not is_threat else
      "brightness(0) saturate(100%) invert(20%) sepia(100%) saturate(6000%) hue-rotate(0deg) brightness(1.2)"
    )

    # --- SENSOR RENDERING ---
    sensor_html = []
    for sensor in sensor_states:
        sx = sensor["x"]
        sy = sensor["y"]
        triggered = sensor["triggered"]

        sensor_color = (
            "rgba(255, 165, 0, 0.16)"
            if triggered else
            "rgba(100, 180, 255, 0.10)"
        )
        sensor_border = (
            "rgba(255, 180, 80, 0.40)"
            if triggered else
            "rgba(120, 190, 255, 0.22)"
        )

        sensor_html.append(f"""
            <div style="
                position: absolute;
                left: {sx}px;
                top: {sy}px;
                width: {2 * SENSOR_RADIUS_PX}px;
                height: {2 * SENSOR_RADIUS_PX}px;
                transform: translate(-50%, -50%);
                border-radius: 50%;
                background: {sensor_color};
                border: 1.5px solid {sensor_border};
                box-shadow: 0 0 12px {sensor_color};
                pointer-events: none;
            "></div>
        """)

    sensors_html = "\n".join(sensor_html)

    return f"""
    <style>
    @keyframes {animation_name} {{
        0%   {{ left: {p1[0]}px; top: {p1[1]}px; }}
        100% {{ left: {p2[0]}px; top: {p2[1]}px; }}
    }}
    </style>

    <div style="
        height: 100%;
        min-height: {IMAGE_HEIGHT_PX + 32}px;
        display: flex;
        align-items: center;
        justify-content: center;
        background: #0f1117;
        border-radius: 18px;
        border: 1px solid #2a2f3a;
        padding: 16px;
        box-sizing: border-box;
    ">
        <div style="
            position: relative;
            width: {IMAGE_WIDTH_PX}px;
            height: {IMAGE_HEIGHT_PX}px;
            min-width: {IMAGE_WIDTH_PX}px;
            max-width: {IMAGE_WIDTH_PX}px;
            min-height: {IMAGE_HEIGHT_PX}px;
            max-height: {IMAGE_HEIGHT_PX}px;
            overflow: hidden;
            border-radius: 18px;
            flex-shrink: 0;
        ">

            <img src="{img_src}" style="
                width: {IMAGE_WIDTH_PX}px;
                height: {IMAGE_HEIGHT_PX}px;
                min-width: {IMAGE_WIDTH_PX}px;
                max-width: {IMAGE_WIDTH_PX}px;
                min-height: {IMAGE_HEIGHT_PX}px;
                max-height: {IMAGE_HEIGHT_PX}px;
                object-fit: fill;
                display: block;
                border-radius: 18px;
            ">

            {sensors_html}

            <img src="{drone_img_src}" style="
                position: absolute;
                left: {p1[0]}px;
                top: {p1[1]}px;
                width: {2 * DRONE_RADIUS_PX * 2}px;
                height: {2 * DRONE_RADIUS_PX * 2}px;
                transform: translate(-50%, -50%);
                animation: {animation_name} {movement_duration}s linear forwards;

                filter: {drone_filter};
                opacity: 0.95;

                pointer-events: none;

            ">

        </div>
    </div>
    """





def build_blank_visual_panel():
    return """
    <div style="
        width: 100%;
        aspect-ratio: 1 / 1;
        min-height: 320px;
        background: #0f1117;
        border: 2px dashed #3b4252;
        border-radius: 18px;
        display: flex;
        align-items: center;
        justify-content: center;
        color: #8b949e;
        font-size: 20px;
        font-weight: 600;
    ">
        Visualization Output
    </div>
    """


# ---- CHATBOT -----
import nest_asyncio
import threading
import asyncio
from mcp.server.fastmcp import FastMCP
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage

nest_asyncio.apply()

mcp = FastMCP("RF_Analysis_Assistant")


@mcp.tool()
def check_current_rf_status() -> str:
    """Reads the current RF environment variables including modulation and frequency hopping."""
    if sys_state.cur_analysis:
        mod = sys_state.cur_analysis["mod"][0]
        freq_hop = str(sys_state.cur_analysis["freq_hop"])
        return f"START_RFT Detected modulation is {mod}. Detected frequency hopping sequence is {freq_hop}. END_RFT"
    return "No RF analysis available. The operator has not scanned a signal yet."

device = "cuda" if torch.cuda.is_available() else "cpu"
@mcp.tool()
def check_current_audio_status() -> str:
  """Checks audio signals to detect whether a drone is present."""
  ckpt = torch.load("/content/drive/MyDrive/ARL/best_cnn.pt", map_location=device)
  classes = ckpt["classes"]
  model = SmallAudioCNN(len(classes)).to(device)
  model.load_state_dict(ckpt["model"])
  model.eval()
  dataset_path = "/content/DroneAudioDataset/Multiclass_Drone_Audio"
  #files = []

  # for c in classes:
  #   folder = os.path.join(dataset_path, c)
  #   for f in os.listdir(folder):
  #     if f.endswith(".wav"):
  #       files.append(os.path.join(folder, f))

  # audio_path = random.choice(files)
  files_threat = []
  files_no_drone = []

  for c in ['membo_1', 'bebop_1']:
    folder = os.path.join(dataset_path, c)
    if os.path.exists(folder):
        files_threat.extend([os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".wav")])

  folder_unknown = os.path.join(dataset_path, 'unknown')
  if os.path.exists(folder_unknown):
      files_no_drone.extend([os.path.join(folder_unknown, f) for f in os.listdir(folder_unknown) if f.endswith(".wav")])

  if IS_DRONE_AUDIO_RANGE:
      audio_path = random.choice(files_threat)
  else:
      audio_path = random.choice(files_no_drone)
  # else:
  #     audio_path = None # Safe fallback

  sys_state.last_audio_path = audio_path

  waveform, sr = torchaudio.load(audio_path)
  Audio(waveform.numpy(), rate=sr)
  waveform = waveform.mean(dim=0, keepdim=True)
  waveform = torchaudio.transforms.Resample(sr,16000)(waveform)

  mel = torchaudio.transforms.MelSpectrogram(
      sample_rate=16000,
      n_fft=1024,
      hop_length=256,
      n_mels=64)

  to_db = torchaudio.transforms.AmplitudeToDB()

  spec = mel(waveform)
  spec = to_db(spec)

  spec = (spec - spec.mean())/(spec.std()+1e-6)

  spec = spec.unsqueeze(0).to(device)


  # Save plot to a base64 string
  buf = io.BytesIO()
  plt.savefig(buf, format='png', pad_inches=5)
  plt.close()
  buf.seek(0)
  img_str = base64.b64encode(buf.read()).decode('utf-8')

  # Store it in sys_state so the Chatbot can grab it
  sys_state.cur_spec_base64 = img_str
  with torch.no_grad():
      logits = model(spec)
      pred = torch.argmax(logits, dim=1).item()

  predicted_class = classes[pred]
  plt.figure(figsize=(5, 5))
  # 'spec' is the tensor from your mel(waveform) call
  plt.imshow(spec.squeeze().cpu().numpy(), origin='lower', aspect='auto', cmap='viridis')
  plt.title(f"Audio Spectrogram:")
  plt.axis('off')
  plt.tight_layout()

  if predicted_class == "unknown":
      return f"Acoustic analysis complete. Drone not detected."
  else:
     return f"Acoustic analysis complete. Drone detected: {predicted_class}."


def start_server():
    print("Starting MCP Server...")
    mcp.run(transport="sse")

server_thread = threading.Thread(target=start_server, name="mcp_server_thread", daemon=True)
server_thread.start()

!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import threading
import subprocess
import time

def run_ollama():
    subprocess.run(["ollama", "serve"])

ollama_thread = threading.Thread(target=run_ollama, daemon=True)
ollama_thread.start()

print("✅ Ollama server is running!")
!ollama pull llama3.2

from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

SYSTEM_PROMPT = (
    "You are an RF engineering assistant in the US army. "
    "You have access to two tools:\n"
    "1. `check_current_rf_status`: Analyzes and provides the currently observed frequency hopping sequence and modulation scheme.\n"
    "2. `check_current_audio_status`: Checks audio signals to detect whether a drone is present. Use this whenever the user asks about audio, acoustics, or drone presence.\n\n"
    "Currently the US army employs the following two modulation and frequency hopping sequences. "
    "That is a legitimate and legal RF signal would either have a frequency hopping sequence of [10,20] and would use BPSK modulation or it would have a frequency hopping sequence of [30,40] with QPSK modulation. "
    "If the modulation scheme is not QPSK or BPSK or if the frequency hopping sequence is not one of [10,20] or [30,40], you must warn the operator immediately. "
    "Based on the results from these tools, your task is to answer the questions so that even a civilian would understand what is going on."
)

global_agent = None


async def get_agent():
    global global_agent
    if global_agent is None:
        client = MultiServerMCPClient({
            "research": {
                "transport": "sse",
                "url": "http://localhost:8000/sse"
            }
        })
        mcp_tools = await client.get_tools()
        global_agent = create_react_agent(llm, mcp_tools)
    return global_agent


async def chat_fn(message, history):
    if not message:
        return "Please enter a question."

    try:
        agent = await get_agent()
        messages = [SystemMessage(content=SYSTEM_PROMPT)]

        if history:
            for msg in history:
                if msg.get("role") == "user":
                    messages.append(HumanMessage(content=msg.get("content", "")))
                elif msg.get("role") == "assistant":
                    messages.append(AIMessage(content=msg.get("content", "")))

        if sys_state.cur_analysis:
            rft_prompt = (
                f"\nSTART_RFT Detected modulation is {sys_state.cur_analysis['mod'][0]}. "
                f"Detected frequency hopping sequence is {str(sys_state.cur_analysis['freq_hop'])}. END_RFT"
            )
            messages.append(HumanMessage(content=message + rft_prompt))
        else:
            messages.append(HumanMessage(content=message))

        response = await agent.ainvoke({"messages": messages})
        return response["messages"][-1].content

    except Exception as e:
        return f"Error connecting to agent: {str(e)}"

async def respond(message, chat_history):
    if chat_history is None: chat_history = []

    # Default blank panel
    vis_html = build_blank_visual_panel()

    try:
        agent = await get_agent()
        messages = [SystemMessage(content=SYSTEM_PROMPT)]
        for msg in chat_history:
            messages.append(AIMessage(content=msg["content"]) if msg["role"] == "assistant" else HumanMessage(content=msg["content"]))

        messages.append(HumanMessage(content=message))
        response = await agent.ainvoke({"messages": messages})
        bot_message = response["messages"][-1].content

        # CHECK IF A SPECTROGRAM WAS GENERATED
        if hasattr(sys_state, 'cur_spec_base64') and sys_state.cur_spec_base64:
            vis_html = f"""
            <div style="width: 100%; aspect-ratio: 1/1; background: #0f1117; border-radius: 18px; overflow: hidden; border: 1px solid #2a2f3a;">
                <img src="data:image/png;base64,{sys_state.cur_spec_base64}" style="width: 100%; height: 100%; object-fit: contain;">
            </div>
            """
            sys_state.cur_spec_base64 = None
        else:
            vis_html = build_blank_visual_panel()

    except Exception as e:
        bot_message = f"Agent Error: {str(e)}"

    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": bot_message})

    # IMPORTANT: Returning 3 values to match msg.submit outputs
    return "", chat_history, vis_html

# --- UI CONTROL FUNCTIONS ---
def refresh_ui(noise):
    generate_mixed_chunk(noise)

    # Keep current analysis in sync with latest chunk.
    # Since the visible spectrum is gone, analyze the full chunk.
    # try:
    #     sys_state.cur_analysis = analyze_selection(0.0, sys_state.chunk_duration)
    # except Exception:
    #     sys_state.cur_analysis = None

    status_html = build_status_circle_html()
    visual_html = build_blank_visual_panel()

    return status_html, visual_html, "Monitoring..."


def stream(noise):
    sys_state.running = True
    while sys_state.running:
        status_html, visual_html, status_text = refresh_ui(noise)
        yield status_html, visual_html, status_text
        time.sleep(2.0)

def pause():
    sys_state.running = False
    return "Paused"

# --- INTERFACE ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📡 RF Interceptor Dashboard")

    with gr.Row():
        # LEFT: status circle only
        with gr.Column(scale=1):
            status_circle = gr.HTML(build_status_circle_html())

            noise = gr.Slider(0, 0.5, 0.1, label="Noise")
            btn_play = gr.Button("▶ Start", variant="primary")
            btn_pause = gr.Button("⏸ Pause", variant="secondary")
            out = gr.HTML("Ready")

        # RIGHT: chatbot on top, blank square below
        with gr.Column(scale=1):
            chatbot = gr.Chatbot(label="ChatBot", height=320)
            msg = gr.Textbox(label="Your Message")
            clear = gr.Button("Clear Memory")
            visual_panel = gr.HTML(build_blank_visual_panel())

    btn_play.click(stream, [noise], [status_circle, visual_panel, out])
    btn_pause.click(pause, None, out)

    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: [], None, chatbot, queue=False)

generate_mixed_chunk(0.1)

if __name__ == "__main__":
    demo.queue().launch(debug=True)

#Multi-RF dataset

In [ ]:
import numpy as np
import pickle

def generate_constellations():
    """Defines and normalizes constellations to unit average power."""
    # BPSK: Average power = 1
    bpsk = np.array([-1, 1]) + 0j

    # QPSK: Average power = 1
    qpsk = np.array([1+1j, 1-1j, -1+1j, -1-1j]) / np.sqrt(2)

    # 16-QAM: alphabet +/- 1, +/- 3. Average power = 10. Normalize by sqrt(10).
    qam16_alphabet = np.array([-3, -1, 1, 3])
    X, Y = np.meshgrid(qam16_alphabet, qam16_alphabet)
    qam16 = (X.flatten() + 1j * Y.flatten()) / np.sqrt(10)

    return bpsk, qpsk, qam16

def generate_interference_dataset(snr_db_list, samples_per_class=1000, sequence_length=128):
    """
    Generates a 6-class dataset for adversarial modulation classification.
    Each sample contains exactly 'sequence_length' symbols (default: 128).
    """
    bpsk, qpsk, qam16 = generate_constellations()

    dataset_X = []
    dataset_y = []
    dataset_snr = []

    for snr in snr_db_list:
        # Calculate noise variance. Signal power for a single Tx is 1.0.
        noise_var = 10 ** (-snr / 10.0)
        noise_std = np.sqrt(noise_var / 2.0) # Divided by 2 for complex noise (I and Q)

        for class_idx in range(6):
            # Vectorized noise generation
            noise = noise_std * (np.random.randn(samples_per_class, sequence_length) +
                                 1j * np.random.randn(samples_per_class, sequence_length))

            # Generate the signal based on the class
            if class_idx == 0:     # No Signal
                signal = np.zeros((samples_per_class, sequence_length), dtype=np.complex128)

            elif class_idx == 1:   # BPSK
                signal = np.random.choice(bpsk, (samples_per_class, sequence_length))

            elif class_idx == 2:   # QPSK
                signal = np.random.choice(qpsk, (samples_per_class, sequence_length))

            elif class_idx == 3:   # 16-QAM
                signal = np.random.choice(qam16, (samples_per_class, sequence_length))

            elif class_idx == 4:   # BPSK + 16-QAM
                sig_user = np.random.choice(bpsk, (samples_per_class, sequence_length))
                sig_adv = np.random.choice(qam16, (samples_per_class, sequence_length))
                signal = sig_user + sig_adv

            elif class_idx == 5:   # QPSK + 16-QAM
                sig_user = np.random.choice(qpsk, (samples_per_class, sequence_length))
                sig_adv = np.random.choice(qam16, (samples_per_class, sequence_length))
                signal = sig_user + sig_adv

            # Receive signal is transmitted signal + AWGN
            rx_signal = signal + noise

            # Convert complex signal into separate I and Q channels for ML models
            # Shape becomes: (samples_per_class, sequence_length, 2)
            iq_signal = np.stack((np.real(rx_signal), np.imag(rx_signal)), axis=-1)

            dataset_X.append(iq_signal)
            dataset_y.append(np.full(samples_per_class, class_idx))
            dataset_snr.append(np.full(samples_per_class, snr))

    # Concatenate all lists into final numpy arrays
    X = np.concatenate(dataset_X, axis=0)
    y = np.concatenate(dataset_y, axis=0)
    snrs = np.concatenate(dataset_snr, axis=0)

    return X, y, snrs

# --- Example Usage ---
if __name__ == "__main__":
    # Test across SNR from -10 dB to 20 dB in steps of 5
    snr_levels = np.arange(-10, 25, 5)

    print("Generating dataset...")
    X, y, snrs = generate_interference_dataset(
        snr_db_list=snr_levels,
        samples_per_class=500,  # 500 samples per class, per SNR level
        sequence_length=128     # Standard length for CNN-based AMC
    )

    print(f"Dataset generated successfully!")
    print(f"Data Shape (X): {X.shape} -> (Total Samples, Time Steps, I/Q Channels)")
    print(f"Labels Shape (y): {y.shape}")

    # ... (previous code generating X, y, snrs) ...

    print("Saving dataset to pickle file...")

    # Bundle the arrays into a dictionary
    dataset_dict = {
        'X': X,
        'y': y,
        'snrs': snrs
    }

    # Write the dictionary to a binary file
    file_name = "rf_interference_dataset.pkl"
    with open(file_name, 'wb') as f:
        pickle.dump(dataset_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"Dataset successfully saved to {file_name}")

IQ-CNN for multi-RF classification

In [ ]:
import torch
import pickle
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

# ---------------------------------------------------------
# 1. Dataset Definition
# ---------------------------------------------------------
class RFInterferenceDataset(Dataset):
    def __init__(self, X, y, snrs):
        # PyTorch Conv1d expects (Batch, Channels, Sequence_Length)
        # Transpose (N, 128, 2) -> (N, 2, 128)
        X_transposed = np.transpose(X, (0, 2, 1))

        self.X = torch.tensor(X_transposed, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.snrs = torch.tensor(snrs, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.snrs[idx]

# ---------------------------------------------------------
# 2. Model Architecture
# ---------------------------------------------------------
class IQ_CNN(nn.Module):
    def __init__(self, num_classes=6):
        super(IQ_CNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv1d(in_channels=2, out_channels=64, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(in_channels=64, out_channels=128, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(in_channels=128, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# ---------------------------------------------------------
# 3. Training Loop (Full Dataset)
# ---------------------------------------------------------
def train_model(model, train_loader, epochs=10, lr=0.001):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels, _ in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_acc = 100 * correct / total

        print(f"Epoch [{epoch+1}/{epochs}] | "
              f"Train Loss: {running_loss/len(train_loader):.4f}, Train Acc: {train_acc:.2f}%")

# ---------------------------------------------------------
# 4. Main Execution and Saving
# ---------------------------------------------------------
def main():
    # ---------------------------------------------------------
    # 1. Load Data from Pickle
    # ---------------------------------------------------------
    print("Loading dataset from pickle file...")
    with open("rf_interference_dataset.pkl", 'rb') as f:
        loaded_data = pickle.load(f)

    # Extract the arrays
    X_data = loaded_data['X']
    y_data = loaded_data['y']
    snrs_data = loaded_data['snrs']

    print(f"Loaded X shape: {X_data.shape}")
    print(f"Loaded y shape: {y_data.shape}")

    # ---------------------------------------------------------
    # 2. Create Dataset and DataLoader
    # ---------------------------------------------------------
    # Instantiate the custom Dataset with the loaded arrays
    full_dataset = RFInterferenceDataset(X_data, y_data, snrs_data)

    # Create the DataLoader to handle batching and shuffling
    batch_size = 64
    train_loader = DataLoader(full_dataset, batch_size=batch_size, shuffle=True)

    # ---------------------------------------------------------
    # 3. Initialize and Train the Model
    # ---------------------------------------------------------
    print("Initializing model...")
    model = IQ_CNN(num_classes=6)

    print("Starting training on 100% of the dataset...")
    # You can adjust epochs and learning rate (lr) as needed
    train_model(model, train_loader, epochs=100, lr=0.001)

    # ---------------------------------------------------------
    # 4. Save the Trained Model Weights
    # ---------------------------------------------------------
    save_path = "rf_interference_iq_cnn_full.pth"
    torch.save(model.state_dict(), save_path)
    print(f"Training complete. Model weights saved successfully to '{save_path}'!")

if __name__ == "__main__":
    main()

Inference for Multi-RF dataset and IQ-CNN

In [ ]:
import gradio as gr
import torch
import torch.nn as nn
import numpy as np
import time
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. IQ-CNN Architecture
# ---------------------------------------------------------
class IQ_CNN(nn.Module):
    def __init__(self, num_classes=6):
        super(IQ_CNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_channels=2, out_channels=64, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(in_channels=64, out_channels=128, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(in_channels=128, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = IQ_CNN(num_classes=6).to(device)

try:
    model.load_state_dict(torch.load("rf_interference_iq_cnn_full.pth", map_location=device))
    print("Model weights loaded successfully.")
except FileNotFoundError:
    print("Warning: rf_interference_iq_cnn_full.pth not found. Using untrained weights.")
model.eval()

# ---------------------------------------------------------
# 2. System State & Signal Generation
# ---------------------------------------------------------
class SystemState:
    def __init__(self):
        self.running = False
        self.current_iq = None
        self.current_classes = [] # Now stores a list of 4 classes
        self.classes = {
            0: "No Signal (AWGN)",
            1: "BPSK (Legitimate User)",
            2: "QPSK (Legitimate User)",
            3: "16-QAM (Adversary Isolated)",
            4: "BPSK + 16-QAM (Colliding)",
            5: "QPSK + 16-QAM (Colliding)"
        }

sys_state = SystemState()

def generate_128_sym_packet(true_class, noise_std):
    """Generates exactly 1 packet (128 symbols)."""
    num_syms = 128
    bpsk = np.random.choice([-1.0, 1.0], num_syms) + 0j

    qpsk_vals = [-1.0, 1.0]
    qpsk = (np.random.choice(qpsk_vals, num_syms) + 1j * np.random.choice(qpsk_vals, num_syms)) / np.sqrt(2)

    qam_vals = [-3.0, -1.0, 1.0, 3.0]
    qam16 = (np.random.choice(qam_vals, num_syms) + 1j * np.random.choice(qam_vals, num_syms)) / np.sqrt(10)

    if true_class == 0:
        clean_signal = np.zeros(num_syms, dtype=complex)
    elif true_class == 1:
        clean_signal = bpsk
    elif true_class == 2:
        clean_signal = qpsk
    elif true_class == 3:
        clean_signal = qam16
    elif true_class == 4:
        clean_signal = bpsk + qam16
    elif true_class == 5:
        clean_signal = qpsk + qam16

    noise = (np.random.randn(num_syms) + 1j * np.random.randn(num_syms)) * noise_std
    noisy_signal = clean_signal + noise

    return np.vstack((noisy_signal.real, noisy_signal.imag)).T

def generate_signal(noise_std):
    """Generates 4 consecutive 128-symbol packets."""
    iq_segments = []
    true_classes = []

    for _ in range(4):
        packet_class = np.random.randint(0, 6)
        true_classes.append(packet_class)
        packet_iq = generate_128_sym_packet(packet_class, noise_std)
        iq_segments.append(packet_iq)

    full_iq_array = np.vstack(iq_segments) # Shape (512, 2)
    return full_iq_array, true_classes

# ---------------------------------------------------------
# 3. Plotting & Inference Logic
# ---------------------------------------------------------
def create_plot(iq_array, start_idx, true_classes):
    """Plots the 512 symbols, coloring each 128-symbol packet based on its class."""
    plt.close('all')
    fig, ax = plt.subplots(figsize=(10, 3.5))

    # Plot each of the 4 packets independently to allow color changes
    for i in range(4):
        start = i * 128
        end = start + 128
        segment = iq_array[start:end]

        is_adversary = true_classes[i] in [3, 4, 5]
        sig_color = 'red' if is_adversary else 'green'

        x_vals = np.arange(start, end)

        # Only add labels on the first iteration to keep the legend clean
        label_I = 'I Component' if i == 0 else ""
        label_Q = 'Q Component' if i == 0 else ""

        ax.plot(x_vals, segment[:, 0], color=sig_color, linestyle='-', alpha=0.9, label=label_I)
        ax.plot(x_vals, segment[:, 1], color=sig_color, linestyle='--', alpha=0.7, label=label_Q)

    # Draw the highlighted sliding window
    ax.axvspan(start_idx, start_idx + 128, color='blue', alpha=0.15, label='128-Sym Window')

    ax.set_xlim(0, 512)
    ax.set_ylim(-3.5, 3.5)
    ax.set_xlabel("Symbol Index")
    ax.set_ylabel("Amplitude")
    ax.legend(loc='upper right')
    plt.tight_layout()
    return fig

def evaluate_window(iq_array, start_idx, true_classes):
    """Extracts the 128 symbols, runs the CNN, and formats the HTML."""
    if iq_array is None:
        return "<div style='color: gray;'>Awaiting data...</div>"

    # Slice exactly 128 symbols for the CNN
    window = iq_array[start_idx : start_idx + 128]

    # Run CNN
    iq_tensor = torch.tensor(window.T, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = model(iq_tensor)
        _, predicted = torch.max(outputs, 1)
        pred_class_idx = predicted.item()

    # Determine which packets the sliding window is overlapping
    start_packet_idx = start_idx // 128
    end_packet_idx = (start_idx + 127) // 128

    if start_packet_idx == end_packet_idx:
        # Perfectly aligned inside one packet
        true_name = f"Aligned with Packet {start_packet_idx + 1}: {sys_state.classes[true_classes[start_packet_idx]]}"
        overlapped_classes = [true_classes[start_packet_idx]]
    else:
        # Straddling two packets
        c1 = sys_state.classes[true_classes[start_packet_idx]]
        c2 = sys_state.classes[true_classes[end_packet_idx]]
        true_name = f"Overlaps Packet {start_packet_idx + 1} ({c1}) and Packet {end_packet_idx + 1} ({c2})"
        overlapped_classes = [true_classes[start_packet_idx], true_classes[end_packet_idx]]

    # Background color logic based on whether an adversary is in ANY of the overlapped packets
    true_adversary_present = any(c in [3, 4, 5] for c in overlapped_classes)
    bg_color = "#ff4c4c" if true_adversary_present else "#4caf50"

    # Warning box based on PREDICTED data
    adversary_warning_text = ""
    if pred_class_idx in [3, 4, 5]:
        adversary_warning_text = "<div style='margin-top: 15px; font-size: 24px; font-weight: bold; color: #fff; background-color: #000; padding: 10px; border-radius: 8px; border: 2px solid yellow; text-align: center;'>⚠️ ADVERSARY DETECTED ⚠️</div>"

    pred_name = sys_state.classes[pred_class_idx]

    html_out = f"""
    <div style="background-color: {bg_color}; padding: 30px; border-radius: 15px; color: white; font-family: sans-serif; box-shadow: 0 4px 8px rgba(0,0,0,0.2);">
        <h3 style="margin-top: 0; margin-bottom: 5px;">Analysis Window [{start_idx} : {start_idx+128}]</h3>
        <div style="font-size: 18px; margin-bottom: 15px;">
            <strong>True Ground Truth:</strong> {true_name}<br>
            <strong>CNN Prediction:</strong> {pred_name}
        </div>
        {adversary_warning_text}
    </div>
    """
    return html_out

# ---------------------------------------------------------
# 4. Gradio Event Handlers
# ---------------------------------------------------------
def start_stream(noise_std):
    sys_state.running = True
    while sys_state.running:
        # Generate 4 packets (512 total symbols)
        iq, true_classes = generate_signal(noise_std)

        # Save to state for when we pause
        sys_state.current_iq = iq
        sys_state.current_classes = true_classes

        # Default start index to 0 while streaming
        fig = create_plot(iq, 0, true_classes)
        html = evaluate_window(iq, 0, true_classes)

        # Forcefully disable/reset the slider to 0 while playing
        yield fig, html, gr.update(value=0, interactive=False)
        time.sleep(1.2)

def pause_stream():
    sys_state.running = False
    # Unlock the slider so the user can sweep across the 4 packets
    return gr.update(interactive=True)

def update_window_on_slider(start_idx):
    if sys_state.current_iq is None:
        return gr.update(), gr.update()

    fig = create_plot(sys_state.current_iq, start_idx, sys_state.current_classes)
    html = evaluate_window(sys_state.current_iq, start_idx, sys_state.current_classes)
    return fig, html

# ---------------------------------------------------------
# 5. UI Layout
# ---------------------------------------------------------
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📡 Live Packetized RF Spectrum Dashboard")
    gr.Markdown("Streams **four 128-symbol packets** (512 symbols total). The signal turns **Red** during an adversary packet, and **Green** during a clean packet. Hit Pause to slide the 128-symbol CNN analysis window across the packets.")

    with gr.Row():
        with gr.Column(scale=2):
            plot_display = gr.Plot(label="Live I/Q Signal")

            # Slider goes up to 384 (so the 128 window ends perfectly at 512)
            window_slider = gr.Slider(minimum=0, maximum=384, step=1, value=0, label="Analysis Window Start Index (Active when Paused)", interactive=False)

            with gr.Row():
                noise_slider = gr.Slider(minimum=0.0, maximum=1.5, value=0.2, step=0.05, label="Noise Level (Std Dev)")
            with gr.Row():
                btn_play = gr.Button("▶ Start Live Stream", variant="primary")
                btn_pause = gr.Button("⏸ Pause & Analyze", variant="stop")

        with gr.Column(scale=1):
            html_display = gr.HTML("<div style='padding: 20px; font-size: 18px; color: gray;'>Awaiting stream...</div>")

    # Wire up the events
    btn_play.click(fn=start_stream, inputs=[noise_slider], outputs=[plot_display, html_display, window_slider])
    btn_pause.click(fn=pause_stream, inputs=None, outputs=[window_slider])
    window_slider.change(fn=update_window_on_slider, inputs=[window_slider], outputs=[plot_display, html_display])

if __name__ == "__main__":
    demo.queue().launch(debug=True)

# Attenuated RF

In [ ]:
import numpy as np
import pickle

def generate_constellations():
    """Defines and normalizes constellations to unit average power."""
    # BPSK: Average power = 1
    bpsk = np.array([-1, 1]) + 0j

    # QPSK: Average power = 1
    qpsk = np.array([1+1j, 1-1j, -1+1j, -1-1j]) / np.sqrt(2)

    # 16-QAM: alphabet +/- 1, +/- 3. Average power = 10. Normalize by sqrt(10).
    qam16_alphabet = np.array([-3, -1, 1, 3])
    X, Y = np.meshgrid(qam16_alphabet, qam16_alphabet)
    qam16 = (X.flatten() + 1j * Y.flatten()) / np.sqrt(10)

    return bpsk, qpsk, qam16

def generate_interference_dataset(snr_db_list, samples_per_class=1000, sequence_length=128):
    """
    Generates a 6-class dataset for adversarial modulation classification.
    Each sample contains exactly 'sequence_length' symbols (default: 128).
    """
    bpsk, qpsk, qam16 = generate_constellations()

    dataset_X = []
    dataset_y = []
    dataset_snr = []

    for snr in snr_db_list:
        # Calculate noise variance. Signal power for a single Tx is 1.0.
        noise_var = 10 ** (-snr / 10.0)
        noise_std = np.sqrt(noise_var / 2.0) # Divided by 2 for complex noise (I and Q)

        for class_idx in range(2):
            # Vectorized noise generation
            noise = noise_std * (np.random.randn(samples_per_class, sequence_length) +
                                 1j * np.random.randn(samples_per_class, sequence_length))

            probs=np.random.uniform(0,1,(samples_per_class))
            merge_mat=np.random.uniform(0,1(samples_per_class,sequence_length))
            merge_mat=(merge_mat<probs).astype(float)

            # Generate the signal based on the class
            if class_idx == 0:     # No Signal
                signal = np.zeros((samples_per_class, sequence_length), dtype=np.complex128)

            elif class_idx == 1:   # BPSK
                signal = np.random.choice(bpsk, (samples_per_class, sequence_length))

            elif class_idx == 2:   # QPSK
                signal = np.random.choice(qpsk, (samples_per_class, sequence_length))

            elif class_idx == 3:   # 16-QAM
                signal = np.random.choice(qam16, (samples_per_class, sequence_length))

            elif class_idx == 4:   # BPSK + 16-QAM
                sig_user = np.random.choice(bpsk, (samples_per_class, sequence_length))
                sig_adv = np.random.choice(qam16, (samples_per_class, sequence_length))
                signal = sig_user + sig_adv

            elif class_idx == 5:   # QPSK + 16-QAM
                sig_user = np.random.choice(qpsk, (samples_per_class, sequence_length))
                sig_adv = np.random.choice(qam16, (samples_per_class, sequence_length))
                signal = sig_user + sig_adv

            # Receive signal is transmitted signal + AWGN
            rx_signal = signal + noise

            # Convert complex signal into separate I and Q channels for ML models
            # Shape becomes: (samples_per_class, sequence_length, 2)
            iq_signal = np.stack((np.real(rx_signal), np.imag(rx_signal)), axis=-1)

            dataset_X.append(iq_signal)
            dataset_y.append(np.full(samples_per_class, class_idx))
            dataset_snr.append(np.full(samples_per_class, snr))

    # Concatenate all lists into final numpy arrays
    X = np.concatenate(dataset_X, axis=0)
    y = np.concatenate(dataset_y, axis=0)
    snrs = np.concatenate(dataset_snr, axis=0)

    return X, y, snrs

# --- Example Usage ---
if __name__ == "__main__":
    # Test across SNR from -10 dB to 20 dB in steps of 5
    snr_levels = np.arange(-10, 25, 5)

    print("Generating dataset...")
    X, y, snrs = generate_interference_dataset(
        snr_db_list=snr_levels,
        samples_per_class=500,  # 500 samples per class, per SNR level
        sequence_length=128     # Standard length for CNN-based AMC
    )

    print(f"Dataset generated successfully!")
    print(f"Data Shape (X): {X.shape} -> (Total Samples, Time Steps, I/Q Channels)")
    print(f"Labels Shape (y): {y.shape}")

    # ... (previous code generating X, y, snrs) ...

    print("Saving dataset to pickle file...")

    # Bundle the arrays into a dictionary
    dataset_dict = {
        'X': X,
        'y': y,
        'snrs': snrs
    }

    # Write the dictionary to a binary file
    file_name = "rf_interference_dataset.pkl"
    with open(file_name, 'wb') as f:
        pickle.dump(dataset_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"Dataset successfully saved to {file_name}")

In [ ]:
  import numpy as np
  probs=np.random.uniform(0,1,(2,1))
  merge_mat=np.random.uniform(0,1,(2,100))
  merge_mat=(merge_mat<probs).astype(float)

In [ ]:
probs

In [ ]:
merge_mat